# Train + eval — 12 models × 2 conditions × 200 epochs, dual test (simulated + real)

Loads the artifacts from the **data-prep notebook** (run that first). Trains every model for **200 epochs, patience 20**, cosine LR with warmup, on the merged ~20k-building set. Evaluates on **both** test sets separately:
- **Simulated** (`Buildings-900K-Test`, 1480 buildings) — both no-weather and +weather checkpoints, since matching synthetic weather exists.
- **Real** (7 open datasets, ~1,900 buildings) — **no-weather checkpoints only** (real buildings don't have reliably matching weather, and the paper's own real-eval doesn't use it either).

### Models (12)
**5 novel** — PWFNet, XPAN, FreqMoE, DRDT, MSCAH (see the earlier notebook for design rationale; unchanged here except `TEMP_IDX` is now set to **0**, confirmed as literal dry-bulb temperature in °C from your own `prepare_buildingsbench.py`, not a guess).
**7 baselines** — naive persistence, linear regression, LSTM, GRU, CNN-LSTM, XGBoost, DLinear (added as a strong, cheap reference).

### Reading the results
Each result table reports **Commercial and Residential NRMSE and CRPS separately** (paper-exact metric). Reference points: paper's best on Buildings-900K-Test = Com 12.91 (no weather, but 900K-pretrained zero-shot — a different data budget than your 20k); last round's TimesNet = Com 12.07 (+weather, 10k budget) — the bar to beat here now that training data has doubled.

### Time
200 epochs × 24 runs (12 models × 2 conditions) is long — resumable by design (`RESET=False` after a disconnect skips finished runs). Real-eval only needs the 12 no-weather checkpoints, so it runs once per model, not per condition.

In [ ]:
# === 1) Mount Drive + GPU + A100 fast-math ===
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > A100."
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True; torch.backends.cudnn.benchmark=True
print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# === 2) All models: 5 novel + 8 baselines (incl. Persistence Ensemble, paper Eq 4; XGBoost path below) ===
import math, torch, torch.nn as nn, torch.nn.functional as F

# ---------------------------------------------------------------------------
# Shared primitives (copied verbatim from the proven pipeline for self-containment)
# ---------------------------------------------------------------------------
def gaussian_nll(mu, raw, target):
    sigma = F.softplus(raw) + 1e-3
    return (0.5*math.log(2*math.pi) + torch.log(sigma) + 0.5*((target-mu)/sigma)**2).mean()

class RevIN(nn.Module):
    """Reversible Instance Normalization. x:(B,L) -> (normalized, mean, std), both (B,1)."""
    def __init__(self): super().__init__()
    def forward(self, x):
        mu = x.mean(1, keepdim=True); sd = x.std(1, keepdim=True).clamp_min(1e-3)
        return (x-mu)/sd, mu, sd

class TimesBlock2(nn.Module):
    """2D temporal conv after folding a 1D series by a fixed period (the mechanism
    that won the previous round, via TimesNet). x:(B,T,D) -> reshape (B,n_cycles,period,D)
    -> Conv2d -> back to (B,T,D). T=sequence length, D=channel dim, period=hours/cycle."""
    def __init__(self, d_model, d_ff, period):
        super().__init__(); self.period = period
        self.conv = nn.Sequential(nn.Conv2d(d_model,d_ff,3,padding=1), nn.GELU(),
                                   nn.Conv2d(d_ff,d_model,3,padding=1))
    def forward(self, x):
        B,T,D = x.shape; p=self.period
        pad=(p-T%p)%p; xp=F.pad(x,(0,0,0,pad)) if pad else x
        n=xp.size(1)//p
        g=xp.reshape(B,n,p,D).permute(0,3,1,2)
        return self.conv(g).permute(0,2,3,1).reshape(B,n*p,D)[:,:T]

# exo layout, guaranteed by the pipeline's gather(): [calendar(n_time) | weather(n_weather)? | building_type(1)]
def split_exo(exo, n_time, n_weather, use_weather):
    cal = exo[..., :n_time]
    if use_weather:
        weather = exo[..., n_time:n_time+n_weather]
    else:
        weather = None
    btype = exo[..., -1:]
    return cal, weather, btype

# ===========================================================================
# 1) PWFNet -- Periodic Weather-response Fusion Network
#    TimesNet backbone (proven) + an explicit dual-balance-point degree-day
#    correction (additive), instead of raw weather fed into attention/MLP.
# ===========================================================================
class PWFNet(nn.Module):
    """
    L, H        : history length (168h), forecast horizon (24h)
    n_time      : width of the calendar block inside exo (6)
    n_weather   : width of the weather block inside exo, if use_weather (7)
    use_weather : whether exo contains a weather block
    temp_idx    : index INTO THE WEATHER BLOCK (0..n_weather-1) that is dry-bulb
                  temperature, if known. If None (default), a learned linear
                  combination of all weather channels is used as a soft
                  temperature proxy (the exact channel order for this dataset
                  wasn't confirmed, so this is the safe default -- pass an int
                  for a more interpretable, literally-physical signal).
    periods     : hour-periods used for 2D folding; (12,24,168) adds a half-day
                  cycle on top of TimesNet's (24,168), aimed at commercial
                  double-peak HVAC schedules.
    """
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=64,d_ff=128,layers=2,periods=(12,24,168),
                 temp_idx=None, dh=64, dropout=0.1, **kw):
        super().__init__()
        self.L,self.H,self.n_time,self.n_weather,self.use_weather=L,H,n_time,n_weather,use_weather
        self.seq=L+H; self.temp_idx=temp_idx
        self.revin=RevIN()
        self.embed=nn.Linear(1,d_model)
        self.pos=nn.Parameter(torch.randn(1,self.seq,d_model)*0.02)
        self.blocks=nn.ModuleList([TimesBlock2(d_model,d_ff,p) for p in periods])
        self.norm=nn.LayerNorm(d_model)
        self.head=nn.Linear(self.seq*d_model, 2*H)
        if use_weather:
            if temp_idx is None: self.temp_proxy=nn.Linear(n_weather,1)
            self.Tbal_heat=nn.Parameter(torch.tensor(-0.3))   # learnable heating balance point (normalized weather units)
            self.Tbal_cool=nn.Parameter(torch.tensor(0.3))    # learnable cooling balance point
            self.dd_mlp=nn.Sequential(nn.Linear(2,dh),nn.GELU(),nn.Linear(dh,1))
    def _degree_day(self, exo):
        if not self.use_weather: return None
        _,weather,_=split_exo(exo,self.n_time,self.n_weather,True)
        temp = weather[...,self.temp_idx] if self.temp_idx is not None else self.temp_proxy(weather).squeeze(-1)
        temp_fut=temp[:,self.L:]                                            # (B,H) only the forecast window matters
        heat=F.relu(self.Tbal_heat-temp_fut); cool=F.relu(temp_fut-self.Tbal_cool)
        return self.dd_mlp(torch.stack([heat,cool],-1)).squeeze(-1)         # (B,H) additive correction
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        full=torch.cat([yn,torch.zeros(B,self.H,device=yh.device,dtype=yn.dtype)],1)
        z=self.embed(full.unsqueeze(-1))+self.pos
        for blk in self.blocks: z=self.norm(z+blk(z))
        out=self.head(z.reshape(B,-1)).view(B,self.H,2)
        mu_n,raw_scale=out[...,0],out[...,1]
        corr=self._degree_day(exo)
        if corr is not None: mu_n=mu_n+corr
        return mu_n,raw_scale,mu,sd

# ===========================================================================
# 2) XPAN -- Cross-Period Phase-Aligned Attention Network
#    Replaces TimesNet's fixed conv kernel with adaptive attention across the
#    same hour-of-day over the past 7 days, and across hours within a day.
# ===========================================================================
class XPAN(nn.Module):
    """Requires L % 24 == 0 (168 -> n_days=7)."""
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=64,heads=4,d_ff=128,dropout=0.1,**kw):
        super().__init__()
        assert L%24==0, "XPAN requires L to be a multiple of 24"
        self.L,self.H,self.n_days=L,H,L//24
        self.n_time,self.n_weather,self.use_weather=n_time,n_weather,use_weather
        c_exo=n_time+(n_weather if use_weather else 0)+1
        self.revin=RevIN()
        self.embed=nn.Linear(1,d_model)
        self.day_pos=nn.Parameter(torch.randn(1,self.n_days,d_model)*0.02)   # position among the 7 days
        self.hour_pos=nn.Parameter(torch.randn(1,24,d_model)*0.02)           # position among the 24 hours
        hl=nn.TransformerEncoderLayer(d_model,heads,d_ff,dropout,batch_first=True,activation='gelu')
        self.hour_phase_enc=nn.TransformerEncoder(hl,1)    # attends across the 7 days, per fixed hour-of-day
        dl=nn.TransformerEncoderLayer(d_model,heads,d_ff,dropout,batch_first=True,activation='gelu')
        self.day_shape_enc=nn.TransformerEncoder(dl,1)     # attends across the 24 hours, per fixed day
        self.fut_proj=nn.Sequential(nn.Linear(H*c_exo,d_model),nn.GELU())
        self.head=nn.Sequential(nn.LayerNorm(3*d_model),nn.Linear(3*d_model,2*H))
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        grid=self.embed(yn.view(B,self.n_days,24).unsqueeze(-1))                # (B,7,24,d_model)
        hp_in=(grid.transpose(1,2)+self.day_pos).reshape(B*24,self.n_days,-1)   # (B*24,7,d_model)
        hp=self.hour_phase_enc(hp_in).mean(1).view(B,24,-1).mean(1)             # (B,d_model)
        ds_in=(grid+self.hour_pos).reshape(B*self.n_days,24,-1)                 # (B*7,24,d_model)
        ds=self.day_shape_enc(ds_in).mean(1).view(B,self.n_days,-1).mean(1)     # (B,d_model)
        fut=self.fut_proj(exo[:,self.L:].reshape(B,-1))                         # (B,d_model)
        out=self.head(torch.cat([hp,ds,fut],-1)).view(B,self.H,2)
        return out[...,0],out[...,1],mu,sd

# ===========================================================================
# 3) FreqMoE -- Frequency/scale-gated Mixture of Experts
#    trend / daily-profile / residual decomposition, each a specialist expert,
#    gated by building-type (+ pooled weather) -- direct response to the
#    paper's finding that commercial and residential behave very differently.
# ===========================================================================
class FreqMoE(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=64,kernel=25,r_patch=24,r_stride=12,dropout=0.1,**kw):
        super().__init__()
        assert L%24==0
        self.L,self.H,self.n_days,self.kernel=L,H,L//24,kernel
        self.n_time,self.n_weather,self.use_weather=n_time,n_weather,use_weather
        self.revin=RevIN()
        self.trend_expert=nn.Linear(L,H)
        self.profile_expert=nn.Sequential(nn.Linear(24,d_model),nn.GELU(),nn.Linear(d_model,H))
        self.r_patch,self.r_stride=r_patch,r_stride
        self.r_np=(L-r_patch)//r_stride+1
        self.resid_embed=nn.Linear(r_patch,d_model)
        self.r_pos=nn.Parameter(torch.randn(1,self.r_np,d_model)*0.02)
        rl=nn.TransformerEncoderLayer(d_model,4,d_model*2,dropout,batch_first=True,activation='gelu')
        self.resid_enc=nn.TransformerEncoder(rl,2)
        self.resid_head=nn.Linear(self.r_np*d_model,H)
        gate_in=1+(n_weather if use_weather else 0)        # building-type (+ pooled weather, if available)
        self.gate=nn.Sequential(nn.Linear(gate_in,32),nn.GELU(),nn.Linear(32,3))
        self.scale_head=nn.Sequential(nn.Linear(3*H,64),nn.GELU(),nn.Linear(64,H))
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        trend=F.avg_pool1d(F.pad(yn.unsqueeze(1),(self.kernel//2,self.kernel//2),mode='replicate'),
                            self.kernel,1).squeeze(1)                          # (B,L) smoothed trend
        daily=yn.view(B,self.n_days,24)
        profile=daily.mean(1)                                                 # (B,24) avg hour-of-day profile
        seasonal=profile.unsqueeze(1).expand(-1,self.n_days,-1).reshape(B,self.L)
        resid=yn-trend-(seasonal-seasonal.mean(1,keepdim=True))               # (B,L) leftover idiosyncratic signal
        e_trend=self.trend_expert(trend)                                      # (B,H)
        e_profile=self.profile_expert(profile)                                # (B,H)
        r=self.resid_embed(resid.unfold(1,self.r_patch,self.r_stride))+self.r_pos
        e_resid=self.resid_head(self.resid_enc(r).reshape(B,-1))              # (B,H)
        cal,weather,btype=split_exo(exo,self.n_time,self.n_weather,self.use_weather)
        gate_in = torch.cat([btype.mean(1), weather.mean(1)],-1) if self.use_weather else btype.mean(1)
        g=F.softmax(self.gate(gate_in),-1)                                    # (B,3) expert weights
        experts=torch.stack([e_trend,e_profile,e_resid],1)                    # (B,3,H)
        mu_n=(g.unsqueeze(-1)*experts).sum(1)
        raw_scale=self.scale_head(experts.reshape(B,-1))
        return mu_n,raw_scale,mu,sd

# ===========================================================================
# 4) DRDT -- Deep Residual Degree-Day Transformer (predict-then-correct)
#    A weather-BLIND PatchTST base (schedule-driven load) + additive degree-
#    day correction -- tests decomposition vs PWFNet's single-stage fusion.
# ===========================================================================
class DRDT(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=96,heads=4,layers=2,d_ff=160,patch=24,stride=12,
                 temp_idx=None,dh=64,dropout=0.1,**kw):
        super().__init__()
        self.L,self.H,self.patch,self.stride=L,H,patch,stride
        self.n_time,self.n_weather,self.use_weather=n_time,n_weather,use_weather
        self.revin=RevIN(); self.np=(L-patch)//stride+1
        self.patch_embed=nn.Linear(patch,d_model)
        self.pos=nn.Parameter(torch.randn(1,self.np,d_model)*0.02)
        enc=nn.TransformerEncoderLayer(d_model,heads,d_ff,dropout,batch_first=True,activation='gelu')
        self.encoder=nn.TransformerEncoder(enc,layers)
        self.cal_fut=nn.Sequential(nn.Linear(H*n_time,d_model),nn.GELU())    # calendar ONLY -- base stays weather-blind
        self.base_head=nn.Sequential(nn.LayerNorm(self.np*d_model+d_model),nn.Linear(self.np*d_model+d_model,2*H))
        self.temp_idx=temp_idx
        if use_weather:
            if temp_idx is None: self.temp_proxy=nn.Linear(n_weather,1)
            self.Tbal_heat=nn.Parameter(torch.tensor(-0.3))
            self.Tbal_cool=nn.Parameter(torch.tensor(0.3))
            self.corr_mlp=nn.Sequential(nn.Linear(2,dh),nn.GELU(),nn.Linear(dh,1))
    def _base(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        z=self.patch_embed(yn.unfold(1,self.patch,self.stride))+self.pos
        z=self.encoder(z).reshape(B,-1)
        cal_future=exo[:,self.L:,:self.n_time].reshape(B,-1)
        z=torch.cat([z,self.cal_fut(cal_future)],-1)
        out=self.base_head(z).view(B,self.H,2)
        return out[...,0],out[...,1],mu,sd
    def _correction(self, exo):
        if not self.use_weather: return None
        _,weather,_=split_exo(exo,self.n_time,self.n_weather,True)
        temp = weather[...,self.temp_idx] if self.temp_idx is not None else self.temp_proxy(weather).squeeze(-1)
        temp_fut=temp[:,self.L:]
        heat=F.relu(self.Tbal_heat-temp_fut); cool=F.relu(temp_fut-self.Tbal_cool)
        return self.corr_mlp(torch.stack([heat,cool],-1)).squeeze(-1)
    def forward(self, yh, exo):
        mu_n,raw_scale,mu,sd=self._base(yh,exo)
        corr=self._correction(exo)
        if corr is not None: mu_n=mu_n+corr
        return mu_n,raw_scale,mu,sd

# ===========================================================================
# 5) MSCAH -- Multi-Scale Conv + Variate Cross-Attention Hybrid
#    TimesNet-style multi-receptive-field conv (query) cross-attends over
#    per-variate tokens (iTransformer-style: load, calendar, weather?, btype).
# ===========================================================================
class _SameConv1d(nn.Module):
    """Conv1d with exact 'same'-length output for ANY kernel size (odd or even).
    nn.Conv1d's symmetric padding=k//2 only preserves length for odd k; for even k
    it over-pads by one (L -> L+1). This pads asymmetrically instead so every
    branch in MSCAH's multi-scale stack outputs exactly length L, regardless of
    kernel parity."""
    def __init__(self, cin, cout, k):
        super().__init__(); self.k=k; self.conv=nn.Conv1d(cin,cout,k,padding=0)
    def forward(self, x):                      # x: (B,cin,L)
        left = (self.k-1)//2; right = self.k-1-left
        return self.conv(F.pad(x,(left,right)))

class MSCAH(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=72,heads=6,d_ff=144,kernels=(3,12,24),dropout=0.1,**kw):
        super().__init__()
        self.L,self.H=L,H; self.n_time,self.n_weather,self.use_weather=n_time,n_weather,use_weather
        self.revin=RevIN()
        self.convs=nn.ModuleList([_SameConv1d(1,d_model,k) for k in kernels])
        self.conv_fuse=nn.Sequential(nn.GELU(),nn.Conv1d(d_model*len(kernels),d_model,1))
        self.conv_pool=nn.AdaptiveAvgPool1d(1)
        self.load_tok=nn.Linear(L,d_model)
        self.cal_tok=nn.Linear(L*n_time,d_model)                              # whole calendar history -> 1 token
        self.weather_tok=nn.Linear(L*n_weather,d_model) if use_weather else None
        self.btype_tok=nn.Linear(1,d_model)
        self.cross=nn.MultiheadAttention(d_model,heads,dropout=dropout,batch_first=True)
        self.cross_norm=nn.LayerNorm(d_model)
        c_exo=n_time+(n_weather if use_weather else 0)+1
        self.fut_proj=nn.Sequential(nn.Linear(H*c_exo,d_model),nn.GELU())
        self.head=nn.Sequential(nn.LayerNorm(2*d_model),nn.Linear(2*d_model,2*H))
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        c=[conv(yn.unsqueeze(1)) for conv in self.convs]
        conv_rep=self.conv_pool(self.conv_fuse(torch.cat(c,1))).squeeze(-1)   # (B,d_model)
        cal,weather,btype=split_exo(exo,self.n_time,self.n_weather,self.use_weather)
        toks=[self.load_tok(yn), self.cal_tok(cal[:,:self.L].reshape(B,-1))]
        if self.use_weather: toks.append(self.weather_tok(weather[:,:self.L].reshape(B,-1)))
        toks.append(self.btype_tok(btype[:,0]))                              # building-type is constant over time
        var_tokens=torch.stack(toks,1)                                       # (B,n_var,d_model)
        a,_=self.cross(conv_rep.unsqueeze(1), var_tokens, var_tokens)
        fused=self.cross_norm(conv_rep+a.squeeze(1))
        fut=self.fut_proj(exo[:,self.L:].reshape(B,-1))
        out=self.head(torch.cat([fused,fut],-1)).view(B,self.H,2)
        return out[...,0],out[...,1],mu,sd

REG={"pwfnet":PWFNet,"xpan":XPAN,"freqmoe":FreqMoE,"drdt":DRDT,"mscah":MSCAH}
def build(name,**kw): return REG[name](**kw)
def count_params(m): return sum(p.numel() for p in m.parameters())
import math, torch, torch.nn as nn, torch.nn.functional as F

# All baselines share the (yh, exo) interface used by the 5 novel models:
#   yh:  (B,L) raw load history          exo: (B,L+H,c_exo) = [calendar|weather?|building_type]

class Naive(nn.Module):
    """Persistence: repeat the load from exactly 24h before each forecast hour (same hour
    yesterday). No learned parameters for the mean; a single learned scalar for uncertainty."""
    def __init__(self, L=168,H=24,**kw):
        super().__init__(); self.L,self.H=L,H
        self.log_sigma=nn.Parameter(torch.zeros(H))
    def forward(self, yh, exo):
        mu = yh[:, self.L-self.H:]                       # (B,H) = load from 24h before the horizon
        mu_n = (mu - yh.mean(1,keepdim=True))/yh.std(1,keepdim=True).clamp_min(1e-3)
        raw = self.log_sigma.unsqueeze(0).expand(yh.size(0),-1)
        return mu_n, raw, yh.mean(1,keepdim=True), yh.std(1,keepdim=True).clamp_min(1e-3)

class LinReg(nn.Module):
    """Paper's 'Linear regression' baseline: one linear layer mapping all L past values
    directly to all H future values, plus future-calendar/weather/building-type context."""
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,dh=64,**kw):
        super().__init__(); self.L,self.H=L,H
        c_exo=n_time+(n_weather if use_weather else 0)+1
        self.mu_lin=nn.Linear(L,H)
        self.fut=nn.Sequential(nn.Linear(H*c_exo,dh),nn.GELU(),nn.Linear(dh,H))
        self.sig=nn.Sequential(nn.Linear(L+H*c_exo,dh),nn.GELU(),nn.Linear(dh,H))
        self.revin=RevIN()
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        fut=exo[:,self.L:].reshape(B,-1)
        mu_n=self.mu_lin(yn)+self.fut(fut)
        raw=self.sig(torch.cat([yn,fut],-1))
        return mu_n,raw,mu,sd

class _RNNBase(nn.Module):
    """Shared encoder-decoder RNN (LSTM/GRU/CNN-LSTM share this scaffold, DeepAR-style:
    RNN encodes L past values, its final hidden state seeds a linear projection to all H
    future steps at once -- avoids autoregressive error compounding for a clean baseline)."""
    def __init__(self, cell, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 hidden=96,layers=2,dropout=0.1,conv_pre=False,**kw):
        super().__init__(); self.L,self.H=L,H
        c_exo=n_time+(n_weather if use_weather else 0)+1
        self.revin=RevIN()
        self.conv_pre = nn.Sequential(nn.Conv1d(1,16,5,padding=2),nn.GELU(),nn.Conv1d(16,1,3,padding=1)) if conv_pre else None
        RNN = {"lstm":nn.LSTM,"gru":nn.GRU}[cell]
        self.rnn=RNN(input_size=1, hidden_size=hidden, num_layers=layers, batch_first=True,
                     dropout=dropout if layers>1 else 0.0)
        self.fut=nn.Sequential(nn.Linear(H*c_exo,hidden),nn.GELU())
        self.head=nn.Sequential(nn.LayerNorm(2*hidden),nn.Linear(2*hidden,2*H))
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        x = yn.unsqueeze(-1)
        if self.conv_pre is not None: x = x + self.conv_pre(yn.unsqueeze(1)).squeeze(1).unsqueeze(-1)
        out,_ = self.rnn(x)                                  # (B,L,hidden)
        h = out[:,-1]                                         # (B,hidden) final encoder state
        fut = self.fut(exo[:,self.L:].reshape(B,-1))
        rep = torch.cat([h,fut],-1)
        o = self.head(rep).view(B,self.H,2)
        return o[...,0],o[...,1],mu,sd

class LSTM(_RNNBase):
    def __init__(self,**kw): super().__init__("lstm",**kw)
class GRU(_RNNBase):
    def __init__(self,**kw): super().__init__("gru",**kw)
class CNNLSTM(_RNNBase):
    def __init__(self,**kw): super().__init__("lstm", conv_pre=True, **kw)

REG_BASE={"naive":Naive,"linreg":LinReg,"lstm":LSTM,"gru":GRU,"cnn_lstm":CNNLSTM}
def build_base(name,**kw): return REG_BASE[name](**kw)
def count_params(m): return sum(p.numel() for p in m.parameters())

class PersistenceEnsemble(nn.Module):
    """Paper Eq 4 exactly: for forecast hour t+i, mu_hat = mean of the load at t+i-24j for
    j=1..7 (same hour, each of the past 7 days); sigma_hat = std of those same 7 values.
    The paper's strongest persistence baseline -- was missing from the roster."""
    def __init__(self, L=168,H=24,**kw):
        super().__init__(); self.L,self.H=L,H
    def forward(self, yh, exo):
        # yh: (B,L) with L=168=7*24 -> reshape to (B,7,24), day-major (day 0 = 7 days ago .. day 6 = yesterday)
        B=yh.size(0); grid=yh.view(B,7,24)               # grid[:,j,h] = load, j days before "today", hour h
        mu = grid.mean(1)                                  # (B,24) = mean over the 7 past days, per hour-of-day
        sd = grid.std(1).clamp_min(1e-3)                   # (B,24) = std over the 7 past days, per hour-of-day
        gmu=yh.mean(1,keepdim=True); gsd=yh.std(1,keepdim=True).clamp_min(1e-3)
        mu_n=(mu-gmu)/gsd
        raw = torch.log(torch.exp(sd/gsd)-1+1e-6)           # invert softplus so softplus(raw)*gsd ~= sd
        return mu_n, raw, gmu, gsd
REG_BASE["persistence_ensemble"]=PersistenceEnsemble
import torch, torch.nn as nn, torch.nn.functional as F

class DLinear(nn.Module):
    """Decomposition (moving-average trend + seasonal residual) + linear -- a paper-relevant,
    hard-to-beat floor. Adapted to the shared n_time/n_weather/use_weather constructor."""
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,kernel=25,dh=128,dropout=0.1,**kw):
        super().__init__(); self.L,self.H,self.k=L,H,kernel; self.revin=RevIN()
        c_exo=n_time+(n_weather if use_weather else 0)+1
        self.trend=nn.Linear(L,H); self.seasonal=nn.Linear(L,H)
        self.fut=nn.Sequential(nn.Linear(H*c_exo,dh),nn.GELU(),nn.Linear(dh,H))
        self.sig=nn.Sequential(nn.Linear(L+H*c_exo,dh),nn.GELU(),nn.Linear(dh,H))
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        tr=F.avg_pool1d(F.pad(yn.unsqueeze(1),(self.k//2,self.k//2),mode='replicate'),self.k,1).squeeze(1)
        se=yn-tr
        fut=exo[:,self.L:].reshape(B,-1)
        mu_n=self.trend(tr)+self.seasonal(se)+self.fut(fut)
        return mu_n, self.sig(torch.cat([yn,fut],-1)), mu, sd

# --- unified registry: 5 novel + 7 baselines (naive,linreg,lstm,gru,cnn_lstm,persistence_ensemble,dlinear).
#     XGBoost is handled separately below (not a torch.nn.Module). ---
REG = {**{"pwfnet":PWFNet,"xpan":XPAN,"freqmoe":FreqMoE,"drdt":DRDT,"mscah":MSCAH}, **REG_BASE, "dlinear":DLinear}
NEURAL_MODELS = list(REG.keys())
ALL_MODELS = NEURAL_MODELS + ["xgboost"]
def build(name, **kw): return REG[name](**kw)
def count_params(m): return sum(p.numel() for p in m.parameters())
print(f"model registry ready: {len(NEURAL_MODELS)} neural + xgboost = {len(ALL_MODELS)} total")

model registry ready: 12 neural + xgboost = 13 total


In [ ]:
# === 3) Shared pipeline: kW inversion, exo normalization, RANDOMIZED validation split, cosine-LR training, evaluation (returns per-building arrays for CI/significance) ===
# === Shared pipeline: kW inversion + exo normalization + building-type covariate + cosine-LR
#     training + evaluation, for FIXED-length (8760h) data -- used for BOTH the 20k train set
#     (compressed artifacts from the data-prep notebook) and the existing simulated test set
#     (already correctly built by prepare_buildingsbench.py -- loaded as-is, untouched). ===
import os, json, pickle, gzip, math, numpy as np, pandas as pd, torch, torch.nn.functional as F
L,H,WIN,HOURS = 168,24,192,8760

def _col(df,opts,what):
    for o in opts:
        if o in df.columns: return o
    raise KeyError(f"no {what} col; have {list(df.columns)}")

def _invert_to_kw(loads, tobj, offset=1.0, verbose=False):
    N=loads.shape[0]; out=np.empty_like(loads,dtype=np.float64)
    def get(i):
        if isinstance(tobj,dict): return tobj.get(i,tobj.get(str(i)))
        if hasattr(tobj,"inverse_transform"): return tobj
        return tobj[i]
    for i in range(N):
        t=get(i)
        if t is None or not hasattr(t,"inverse_transform"): return None
        with np.errstate(over="ignore", invalid="ignore"):
            out[i]=np.asarray(t.inverse_transform(loads[i].reshape(-1,1))).ravel()
    out=out-offset; bad=~np.isfinite(out)
    if bad.any():
        out[bad]=np.nan; out=np.clip(out,0.0,None)
        med=np.nanmedian(out,axis=1,keepdims=True); med=np.where(np.isfinite(med),med,0.0)
        out=np.where(np.isnan(out),med,out)
        if verbose: print(f"  [transform] {100*bad.mean():.3f}% overflow -> per-building median")
    else: out=np.clip(out,0.0,None)
    return out.astype(np.float32)

def _finish_ds(loads, tf, man, wuniq_map, kw_ok, dev, exo_stats, verbose):
    keys=np.array([f"{r}__{c}" for r,c in zip(man["release"].astype(str), man["county"].astype(str))])
    uk=sorted(set(k for k in keys if k.split("__")[1] in wuniq_map))
    kid={k:i for i,k in enumerate(uk)}
    wuniq=np.stack([wuniq_map[k.split("__")[1]] for k in uk]).astype(np.float32)
    keep=np.array([k in kid for k in keys])
    if not keep.all() and verbose: print(f"  [warn] dropping {(~keep).sum()} buildings w/o matching weather")
    loads=loads[keep]; keys=keys[keep]; rels=man["release"].astype(str).values[keep]
    b2w=np.array([kid[k] for k in keys]); is_res=np.array(["resstock" in r for r in rels])
    btype=np.where(is_res,-1.0,1.0).astype(np.float32)
    if exo_stats is None:
        wm=wuniq.reshape(-1,wuniq.shape[2]).mean(0); ws=wuniq.reshape(-1,wuniq.shape[2]).std(0); ws[ws<1e-6]=1.0
        tm=tf.mean(0); ts=tf.std(0); ts[ts<1e-6]=1.0; exo_stats=(wm,ws,tm,ts)
    wm,ws,tm,ts=exo_stats
    wuniq_n=((wuniq-wm)/ws).astype(np.float32); tf_n=((tf-tm)/ts).astype(np.float32)
    T=lambda a,dt: torch.tensor(a,dtype=dt,device=dev)
    return dict(loads=T(loads,torch.float32),tf=T(tf_n,torch.float32),wuniq=T(wuniq_n,torch.float32),
                b2w=T(b2w,torch.long),is_res=T(is_res,torch.bool),btype=T(btype,torch.float32),
                N=loads.shape[0],Ft=tf.shape[1],Fw=wuniq.shape[2],kw_ok=kw_ok,exo_stats=exo_stats)

def load_ds_existing(root, dev, verbose=False, exo_stats=None):
    """Load the ORIGINAL per-file format (bench_data / bench_data_test), UNCHANGED from the
    already-working pipeline -- used for the simulated test set, which prepare_buildingsbench.py
    already builds correctly (TEST_BASE is a separate, dedicated root -- nothing to redo here)."""
    loads=np.load(f"{root}/loads.npy"); tf=np.load(f"{root}/time_feats.npy")
    man=pd.read_parquet(f"{root}/manifest.parquet")
    rc=_col(man,["release","dataset","upgrade"],"release"); cc=_col(man,["county","in.county"],"county")
    man=man.rename(columns={rc:"release",cc:"county"})
    kw_ok=False; tp=f"{root}/transformers.pkl"
    if os.path.exists(tp):
        try:
            tobj=pickle.load(open(tp,"rb")); off=1.0
            try: off=float(json.load(open(f'{root}/config.json')).get('load_offset',1.0))
            except Exception: pass
            inv=_invert_to_kw(loads,tobj,off,verbose)
            if inv is not None and np.isfinite(inv).all() and np.nanmedian(inv)>0:
                if verbose: print(f"  [transform] inverted to kW (type={type(tobj).__name__})"); loads=inv; kw_ok=True
        except Exception as e:
            if verbose: print(f"  [transform] failed: {type(e).__name__}")
    wmap={}
    for f in os.listdir(f"{root}/weather"):
        if f.endswith(".npy"):
            county=f[:-4].split("__")[-1]; wmap[county]=np.load(f"{root}/weather/{f}")
    return _finish_ds(loads, tf, man, wmap, kw_ok, dev, exo_stats, verbose)

def load_ds_compressed(root, dev, verbose=False, exo_stats=None):
    """Load the COMPRESSED artifacts produced by the data-prep notebook (20k-building train set)."""
    d=np.load(f"{root}/loads_timefeats.npz"); loads, tf = d["loads"], d["time_feats"]
    man=pd.read_parquet(f"{root}/manifest.parquet")
    with gzip.open(f"{root}/transformers.pkl.gz","rb") as f: tobj=pickle.load(f)
    off=json.load(open(f"{root}/config.json")).get("load_offset",1.0)
    inv=_invert_to_kw(loads,tobj,off,verbose)
    kw_ok=False
    if inv is not None and np.isfinite(inv).all() and np.nanmedian(inv)>0:
        if verbose: print(f"  [transform] inverted to kW"); loads=inv; kw_ok=True
    wz=np.load(f"{root}/weather.npz"); wmap={k[2:]:wz[k] for k in wz.files}
    return _finish_ds(loads, tf, man, wmap, kw_ok, dev, exo_stats, verbose)

def gather(ds,b,s,use_w):
    idx=s[:,None]+torch.arange(WIN,device=b.device)
    win=ds["loads"][b[:,None].expand(-1,WIN),idx]; xm=ds["tf"][idx]
    parts=[xm]
    if use_w:
        wall=ds["wuniq"][ds["b2w"][b]]; parts.append(torch.gather(wall,1,idx.unsqueeze(-1).expand(-1,-1,ds["Fw"])))
    parts.append(ds["btype"][b].view(-1,1,1).expand(-1,WIN,1))
    return win[:,:L], win[:,L:], torch.cat(parts,-1)

def train(model, base, ds, dev, use_w, epochs=200, patience=20, steps=300, bs=512, lr=4e-4,
          vfrac=0.05, amp=True, amp_dtype=torch.bfloat16, clip=1.0, warmup_frac=0.03, log=False):
    try: opt=torch.optim.AdamW(base.parameters(),lr=lr,weight_decay=1e-4,fused=(dev=="cuda"))
    except TypeError: opt=torch.optim.AdamW(base.parameters(),lr=lr,weight_decay=1e-4)
    # shuffle building order ONCE before splitting -- taking a plain array-tail as validation would
    # bias it toward whichever counties/buildings happened to be appended last during data prep
    Nv=max(1,int(ds["N"]*vfrac))
    _perm=torch.randperm(ds["N"],generator=torch.Generator().manual_seed(0),device="cpu").to(dev)
    val_b=_perm[:Nv]; train_pool=_perm[Nv:]
    tot=epochs*steps; warm=max(50,int(tot*warmup_frac)); gstep=0
    def setlr(g):
        f = g/warm if g<warm else 0.5*(1+math.cos(math.pi*(g-warm)/max(1,tot-warm)))
        for pg in opt.param_groups: pg["lr"]=lr*f
    best=1e9; bstate=None; bad=0; smax=HOURS-WIN
    ac=lambda: torch.autocast("cuda",dtype=amp_dtype,enabled=(amp and dev=="cuda"))
    for ep in range(epochs):
        model.train()
        for _ in range(steps):
            setlr(gstep); gstep+=1
            b=train_pool[torch.randint(0,train_pool.numel(),(bs,),device=dev)]
            s=torch.randint(0,smax,(bs,),device=dev)
            yh,yf,exo=gather(ds,b,s,use_w)
            with ac(): mu_n,rs,mean,sd=model(yh,exo)
            tn=(yf-mean.float())/sd.float()
            loss=gaussian_nll(mu_n.float(),rs.float(),tn)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(base.parameters(),clip); opt.step()
        model.eval(); vs=[]
        with torch.no_grad():
            for v0 in range(0,len(val_b),128):
                b=val_b[v0:v0+128]; s=torch.full((len(b),),smax//2,device=dev)
                yh,yf,exo=gather(ds,b,s,use_w)
                with ac(): mu_n,rs,mean,sd=model(yh,exo)
                vs.append(gaussian_nll(mu_n.float(),rs.float(),(yf-mean.float())/sd.float()).item())
        v=float(np.mean(vs))
        if v<best-1e-4: best=v; bstate={k:t.detach().clone() for k,t in base.state_dict().items()}; bad=0
        else: bad+=1
        if log and ep%10==0: print(f"    ep{ep:03d} val={v:.4f} best={best:.4f} bad={bad}")
        if bad>=patience:
            if log: print(f"    early stop @ep{ep}")
            break
    if bstate: base.load_state_dict(bstate)
    return best

def ncdf(x): return 0.5*(1+torch.erf(x/math.sqrt(2)))
def npdf(x): return torch.exp(-0.5*x*x)/math.sqrt(2*math.pi)
@torch.no_grad()
def evaluate(model,ds,dev,use_w,stride=24,chunk=1024,amp=True,amp_dtype=torch.bfloat16):
    """Returns a dict with BOTH the summary (median) stats used for the results table, AND the
    raw per-building arrays (nrmse, nmae, crps, is_res) needed for bootstrap CIs and paired
    significance tests -- point estimates alone aren't enough to claim one model beats another."""
    model.eval(); ac=lambda: torch.autocast("cuda",dtype=amp_dtype,enabled=(amp and dev=="cuda"))
    starts=list(range(0,HOURS-WIN,stride)); N=ds["N"]
    se=torch.zeros(N,device=dev); sae=torch.zeros(N,device=dev)
    sy=torch.zeros(N,device=dev); sy2=torch.zeros(N,device=dev)
    cnt=torch.zeros(N,device=dev); cr=torch.zeros(N,device=dev)
    pairs=[(b,s) for b in range(N) for s in starts]
    for i in range(0,len(pairs),chunk):
        ch=pairs[i:i+chunk]
        b=torch.tensor([p[0] for p in ch],device=dev); s=torch.tensor([p[1] for p in ch],device=dev)
        yh,yf,exo=gather(ds,b,s,use_w)
        with ac(): mu_n,rs,mean,sd=model(yh,exo)
        mu_n,rs,mean,sd=mu_n.float(),rs.float(),mean.float(),sd.float()
        mu=mu_n*sd+mean; sigma=(F.softplus(rs)+1e-3)*sd
        se.index_add_(0,b,((mu-yf)**2).sum(1)); sae.index_add_(0,b,(mu-yf).abs().sum(1))
        sy.index_add_(0,b,yf.sum(1)); sy2.index_add_(0,b,(yf**2).sum(1))
        cnt.index_add_(0,b,torch.full_like(se[b],float(H)))
        z=(yf-mu)/sigma
        cr.index_add_(0,b,(sigma*(z*(2*ncdf(z)-1)+2*npdf(z)-1/math.sqrt(math.pi))).sum(1))
    rmse=torch.sqrt(se/cnt); mae=sae/cnt; meany=sy/cnt; stdy=torch.sqrt((sy2/cnt-meany**2).clamp_min(1e-8))
    denom=meany if ds["kw_ok"] else stdy
    nrmse=(100*rmse/denom).cpu().numpy(); nmae=(100*mae/denom).cpu().numpy()
    crps=(cr/cnt).cpu().numpy(); res=ds["is_res"].cpu().numpy(); out={}
    for lab,m in [("Com",~res),("Res",res)]:
        if m.any():
            out[f"{lab} NRMSE"]=float(np.median(nrmse[m])); out[f"{lab} NMAE"]=float(np.median(nmae[m]))
            out[f"{lab} CRPS"]=float(np.median(crps[m]))
    out["_nrmse"]=nrmse; out["_nmae"]=nmae; out["_crps"]=crps; out["_is_res"]=res  # raw per-building, for CI/significance
    return out

def gaussian_nll(mu, raw, target):
    sigma = F.softplus(raw) + 1e-3
    return (0.5*math.log(2*math.pi) + torch.log(sigma) + 0.5*((target-mu)/sigma)**2).mean()
print("shared pipeline ready")

def load_real_set(in_path):
    # in_path: the .pkl.gz file Notebook A saved (REAL_TEST_PATH). Returns a list of dicts:
    # building_id (str), building_type (+1.0=commercial/-1.0=residential), series (pd.Series, kW, DatetimeIndex)
    import gzip, pickle
    with gzip.open(in_path,"rb") as f: packed=pickle.load(f)
    out=[]
    for b in packed:
        s=pd.Series(b["values"], index=pd.to_datetime(b["index"]))
        out.append(dict(building_id=b["building_id"], building_type=b["building_type"], series=s))
    return out

shared pipeline ready


In [ ]:
# === XGBoost path (multi-output, GPU) -- same (yh,exo) convention as the neural models ===
import numpy as np, math, torch

def _xgb_xy(yh, exo, L):
    mu=yh.mean(1,keepdim=True); sd=yh.std(1,keepdim=True).clamp_min(1e-3)
    yn=(yh-mu)/sd; H=exo.size(1)-L
    parts=[yn, exo[:,L:].reshape(yh.size(0),-1)]
    return (torch.cat(parts,-1).cpu().numpy(), mu.squeeze(1).cpu().numpy(), sd.squeeze(1).cpu().numpy())

def xgb_fit_sim(gather_fn, ds, dev, use_w, L, H, n_windows=60000, device="cpu", n_estimators=400, max_depth=6, hours=8760, win=192):
    import xgboost as xgb
    smax=hours-win
    b=torch.randint(0,ds["N"],(n_windows,),device=dev); s=torch.randint(0,smax,(n_windows,),device=dev)
    yh,yf,exo=gather_fn(ds,b,s,use_w)
    X,mu,sd=_xgb_xy(yh,exo,L)
    yf_np = yf.cpu().numpy()                                    # convert BEFORE arithmetic -- mu,sd are already numpy
    Y = (yf_np - mu.reshape(-1,1)) / sd.reshape(-1,1)
    k=int(len(X)*0.9)
    p=dict(n_estimators=n_estimators,max_depth=max_depth,learning_rate=0.1,subsample=0.8,
           colsample_bytree=0.8,tree_method="hist",device=device)
    try:
        m=xgb.XGBRegressor(multi_strategy="multi_output_tree",**p); m.fit(X[:k],Y[:k])
    except Exception:
        from sklearn.multioutput import MultiOutputRegressor
        m=MultiOutputRegressor(xgb.XGBRegressor(**p)); m.fit(X[:k],Y[:k])
    sigma=(Y[k:]-m.predict(X[k:])).std(0)
    return m, np.maximum(sigma,1e-3)

def xgb_eval_sim(model, sigma, gather_fn, ds, dev, use_w, L, H, kw_ok, stride=24, chunk=8192, hours=8760, win=192):
    starts=list(range(0,hours-win,stride)); N=ds["N"]
    se=np.zeros(N); sae=np.zeros(N); sy=np.zeros(N); sy2=np.zeros(N); cnt=np.zeros(N); cr=np.zeros(N)
    pairs=[(b,s) for b in range(N) for s in starts]
    for i in range(0,len(pairs),chunk):
        ch=pairs[i:i+chunk]; bidx=np.array([p[0] for p in ch])
        b=torch.tensor(bidx,device=dev); s=torch.tensor([p[1] for p in ch],device=dev)
        yh,yf,exo=gather_fn(ds,b,s,use_w)
        X,mu,sd=_xgb_xy(yh,exo,L); P=model.predict(X)
        yhat=P*sd[:,None]+mu[:,None]; ytrue=yf.cpu().numpy()
        np.add.at(se,bidx,((yhat-ytrue)**2).sum(1)); np.add.at(sae,bidx,np.abs(yhat-ytrue).sum(1))
        np.add.at(sy,bidx,ytrue.sum(1)); np.add.at(sy2,bidx,(ytrue**2).sum(1)); np.add.at(cnt,bidx,H)
        sr=sigma[None,:]*sd[:,None]; z=(ytrue-yhat)/sr
        Phi=np.vectorize(lambda v: 0.5*(1+math.erf(v/math.sqrt(2))))(z)
        phi=np.exp(-0.5*z*z)/math.sqrt(2*math.pi)
        np.add.at(cr,bidx,(sr*(z*(2*Phi-1)+2*phi-1/math.sqrt(math.pi))).sum(1))
    rmse=np.sqrt(se/cnt); mae=sae/cnt; meany=sy/cnt; stdy=np.sqrt(np.clip(sy2/cnt-meany**2,1e-8,None))
    denom=meany if kw_ok else stdy
    nrmse=100*rmse/denom; nmae=100*mae/denom; crps=cr/cnt
    isr=ds["is_res"].cpu().numpy(); out={}
    for lab,mask in [("Com",~isr),("Res",isr)]:
        if mask.any():
            out[f"{lab} NRMSE"]=float(np.median(nrmse[mask])); out[f"{lab} NMAE"]=float(np.median(nmae[mask]))
            out[f"{lab} CRPS"]=float(np.median(crps[mask]))
    out["_nrmse"]=nrmse; out["_nmae"]=nmae; out["_crps"]=crps; out["_is_res"]=isr
    return out
print("xgboost path ready")

xgboost path ready


In [ ]:
# === 3c) Real-building evaluation (variable-length windowing, no weather, NMAE + per-building arrays) ===
import numpy as np

def calendar_features(index):
    """Paper Sec 3.1: hour-of-day, day-of-week, day-of-year, each cyclically sin/cos encoded.
    index: a pandas DatetimeIndex of any length/start date. Returns (len(index), 6) float32:
    [sin(hour), cos(hour), sin(dow), cos(dow), sin(doy), cos(doy)]. Using REAL calendar values
    (not a sequential position index) is what makes this valid for buildings starting on ANY
    date -- essential for the real-eval set, and paper-exact for the simulated set."""
    hour = index.hour.values.astype(np.float32)          # 0-23
    dow  = index.dayofweek.values.astype(np.float32)      # 0-6 (Monday=0)
    doy  = index.dayofyear.values.astype(np.float32)      # 1-366
    return np.stack([np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24),
                      np.sin(2*np.pi*dow/7),  np.cos(2*np.pi*dow/7),
                      np.sin(2*np.pi*doy/365.25), np.cos(2*np.pi*doy/365.25)], 1).astype(np.float32)   # identical to prepare_buildingsbench.py: Box-Cox needs strictly positive input


# === Real-building evaluation: variable-length per-building windowing (no fixed 8760 grid,
#     no weather -- real buildings' locations/instrumentation don't give reliable matching
#     weather, and the paper's own real-eval doesn't use it either). Uses the SAME
#     calendar_features() formula as training so the no-weather checkpoint's calendar
#     encoding means the same thing here as it did during training. ===
import math, numpy as np, torch, torch.nn.functional as F

L, H, WIN = 168, 24, 192

@torch.no_grad()
def evaluate_real(model, buildings, dev, stride=24, max_windows_per_building=90, batch=2048, amp=True, amp_dtype=torch.bfloat16):
    """buildings: list of {building_id, building_type(+1/-1), series: pd.Series(kW, DatetimeIndex)}.
    Returns summary medians (Com/Res NRMSE, NMAE, CRPS) PLUS raw per-building arrays
    (_nrmse,_nmae,_crps,_is_res,_building_ids) for bootstrap CIs / paired significance tests."""
    model.eval()
    ac = lambda: torch.autocast("cuda", dtype=amp_dtype, enabled=(amp and dev=="cuda"))
    # ---- gather all (building, window) start-positions up front (bounded per building) ----
    jobs=[]   # (b_idx, start)
    per_building_windows={}
    for bi, b in enumerate(buildings):
        n=len(b["series"])
        if n < WIN: continue
        starts=list(range(0, n-WIN, stride))
        if len(starts) > max_windows_per_building:
            rng=np.random.default_rng(bi)
            starts=sorted(rng.choice(starts, max_windows_per_building, replace=False).tolist())
        per_building_windows[bi]=starts
        jobs += [(bi,s) for s in starts]
    if not jobs: return {"n_buildings": 0}

    se=np.zeros(len(buildings)); sae=np.zeros(len(buildings)); sy=np.zeros(len(buildings)); sy2=np.zeros(len(buildings))
    cnt=np.zeros(len(buildings)); cr=np.zeros(len(buildings))
    def ncdf(x): return 0.5*(1+torch.erf(x/math.sqrt(2)))
    def npdf(x): return torch.exp(-0.5*x*x)/math.sqrt(2*math.pi)

    for i in range(0, len(jobs), batch):
        chunk=jobs[i:i+batch]
        yh_list, exo_list, yf_list, bidx = [], [], [], []
        for bi, s in chunk:
            b=buildings[bi]; ser=b["series"]
            win=ser.values[s:s+WIN].astype(np.float32)
            idx=ser.index[s:s+WIN]
            cal=calendar_features(idx)                                     # (WIN,6)
            btype_col=np.full((WIN,1), b["building_type"], dtype=np.float32)
            exo=np.concatenate([cal, btype_col],1)                          # (WIN,7) == n_time(6)+btype(1), no-weather width
            yh_list.append(win[:L]); yf_list.append(win[L:]); exo_list.append(exo); bidx.append(bi)
        yh=torch.tensor(np.stack(yh_list),device=dev); yf=torch.tensor(np.stack(yf_list),device=dev)
        exo=torch.tensor(np.stack(exo_list),device=dev)
        with ac(): mu_n,rs,mean,sd=model(yh,exo)
        mu_n,rs,mean,sd=mu_n.float(),rs.float(),mean.float(),sd.float()
        mu=mu_n*sd+mean; sigma=(F.softplus(rs)+1e-3)*sd
        err=((mu-yf)**2).sum(1).cpu().numpy(); aerr=(mu-yf).abs().sum(1).cpu().numpy()
        ysum=yf.sum(1).cpu().numpy(); y2sum=(yf**2).sum(1).cpu().numpy()
        z=(yf-mu)/sigma
        crps=(sigma*(z*(2*ncdf(z)-1)+2*npdf(z)-1/math.sqrt(math.pi))).sum(1).cpu().numpy()
        for j,bi in enumerate(bidx):
            se[bi]+=err[j]; sae[bi]+=aerr[j]; sy[bi]+=ysum[j]; sy2[bi]+=y2sum[j]; cnt[bi]+=H; cr[bi]+=crps[j]

    valid = cnt > 0
    rmse=np.zeros_like(se); mae=np.zeros_like(sae); meany=np.zeros_like(sy); crps_avg=np.zeros_like(cr)
    rmse[valid]=np.sqrt(se[valid]/cnt[valid]); mae[valid]=sae[valid]/cnt[valid]
    meany[valid]=sy[valid]/cnt[valid]; crps_avg[valid]=cr[valid]/cnt[valid]
    ok = valid & (meany>1e-6)
    nrmse=np.full(len(buildings), np.nan); nmae=np.full(len(buildings), np.nan)
    nrmse[ok]=100*rmse[ok]/meany[ok]; nmae[ok]=100*mae[ok]/meany[ok]
    btypes=np.array([b["building_type"] for b in buildings])
    out={"n_buildings": int(ok.sum())}
    for lab, mask in [("Com", (btypes>0)&ok), ("Res", (btypes<0)&ok)]:
        if mask.any():
            out[f"{lab} NRMSE"]=float(np.nanmedian(nrmse[mask])); out[f"{lab} NMAE"]=float(np.nanmedian(nmae[mask]))
            out[f"{lab} CRPS"]=float(np.median(crps_avg[mask]))
    out["_nrmse"]=nrmse; out["_nmae"]=nmae; out["_crps"]=crps_avg; out["_is_res"]=(btypes<0); out["_valid"]=ok
    return out
print("real-eval pipeline ready")

real-eval pipeline ready


In [ ]:
# === Statistical rigor: bootstrap CI (matches the paper's own "95% stratified bootstrap CI
#     for the median") + paired significance testing (so "beats X" is a tested claim, not just
#     a smaller point estimate). ===
import numpy as np
from scipy.stats import wilcoxon

def bootstrap_median_ci(values, n_boot=2000, ci=95, seed=0):
    """values: 1D array of a per-building metric (e.g. NRMSE for all commercial buildings).
    Resamples `values` WITH replacement n_boot times, takes the median of each resample, and
    returns (point_median, ci_low, ci_high) at the given confidence level -- e.g. ci=95 gives
    the 2.5th/97.5th percentile of the bootstrap distribution of the median."""
    values = np.asarray(values); values = values[~np.isnan(values)]
    if len(values) == 0: return (float("nan"), float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    boots = np.array([np.median(rng.choice(values, size=len(values), replace=True)) for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [(100-ci)/2, 100-(100-ci)/2])
    return (float(np.median(values)), float(lo), float(hi))

def paired_significance(values_a, values_b, alpha=0.05):
    """Paired Wilcoxon signed-rank test: is model A's per-building metric significantly
    different from model B's, on the SAME buildings (values_a[i] and values_b[i] must be the
    same building i for both arrays)? Returns (p_value, a_better: bool or None).
    a_better=True means A's median is lower AND the difference is significant at `alpha`."""
    a = np.asarray(values_a); b = np.asarray(values_b)
    if a.shape != b.shape:    # e.g. accidentally comparing two different test sets -- can't pair these
        return (float("nan"), None)
    mask = ~(np.isnan(a) | np.isnan(b))
    a, b = a[mask], b[mask]
    if len(a) < 10 or np.allclose(a, b):   # too few paired buildings, or literally identical -> no claim
        return (float("nan"), None)
    try:
        stat, p = wilcoxon(a, b)
    except ValueError:                       # all-zero differences etc.
        return (float("nan"), None)
    a_better = bool(p < alpha and np.median(a) < np.median(b))
    return (float(p), a_better)
print("stats helpers ready")

stats helpers ready


In [ ]:
# === 4) Config ===
import os, glob, torch
BENCH = "/content/drive/MyDrive/quick/bench"
if not os.path.isdir(BENCH):
    h=glob.glob("/content/drive/MyDrive/**/bench_data", recursive=True); BENCH=os.path.dirname(h[0]) if h else BENCH
assert os.path.isdir(BENCH), f"Set BENCH manually; tried {BENCH}"
TRAIN_DIR = f"{BENCH}/expanded_data/train_20k"
SIM_TEST_DIR = f"{BENCH}/bench_data_test"                       # unchanged, existing
REAL_TEST_PATH = f"{BENCH}/expanded_data/real_test/real_buildings.pkl.gz"
assert os.path.isdir(TRAIN_DIR), f"Run the data-prep notebook first -- {TRAIN_DIR} not found"

RESULTS_DIR=f"{BENCH}/novel20k_results"; os.makedirs(RESULTS_DIR,exist_ok=True)
WEIGHTS_DIR=f"{RESULTS_DIR}/weights"; os.makedirs(WEIGHTS_DIR,exist_ok=True)
SIM_CSV=f"{RESULTS_DIR}/sim_results.csv"; REAL_CSV=f"{RESULTS_DIR}/real_results.csv"

EPOCHS, PATIENCE = 200, 20
STEPS, BS, LR = 300, 512, 4e-4
SEED = 0
USE_AMP, AMP_DTYPE = True, torch.bfloat16
RESET = True          # wipe old results on first clean run; False to resume after a disconnect
TEMP_IDX = 0           # CONFIRMED (prepare_buildingsbench.py): weather channel 0 = dry-bulb temp (deg C)

RNN_SET = {"lstm","gru","cnn_lstm"}          # trained fp32 (bf16 destabilizes RNNs, per earlier debugging)

MODEL_KW = {
  "pwfnet":  dict(d_model=64, d_ff=128, layers=2, periods=(12,24,168), temp_idx=TEMP_IDX, dh=64, dropout=0.1),
  "xpan":    dict(d_model=64, heads=4, d_ff=128, dropout=0.1),
  "freqmoe": dict(d_model=64, kernel=25, r_patch=24, r_stride=12, dropout=0.1),
  "drdt":    dict(d_model=96, heads=4, layers=2, d_ff=160, patch=24, stride=12, temp_idx=TEMP_IDX, dh=64, dropout=0.1),
  "mscah":   dict(d_model=72, heads=6, d_ff=144, kernels=(3,12,24), dropout=0.1),
  "naive":   dict(),
  "linreg":  dict(dh=64),
  "lstm":    dict(hidden=96, layers=2, dropout=0.1),
  "gru":     dict(hidden=96, layers=2, dropout=0.1),
  "cnn_lstm":dict(hidden=96, layers=2, dropout=0.1),
  "dlinear": dict(kernel=25, dh=128, dropout=0.1),
  "persistence_ensemble": dict(),   # paper Eq 4, zero learnable parameters -- not trained, closed-form
}
XGB_WINDOWS, XGB_TREES, XGB_DEPTH = 80000, 500, 6
ALL_MODELS = list(MODEL_KW.keys()) + ["xgboost"]
CONDITIONS = {"A": False, "B": True}
print(f"BENCH={BENCH}\nmodels ({len(ALL_MODELS)}): {ALL_MODELS}\n{EPOCHS} ep/patience {PATIENCE} | TEMP_IDX={TEMP_IDX}")

BENCH=/content/drive/MyDrive/quick/bench
models (13): ['pwfnet', 'xpan', 'freqmoe', 'drdt', 'mscah', 'naive', 'linreg', 'lstm', 'gru', 'cnn_lstm', 'dlinear', 'persistence_ensemble', 'xgboost']
200 ep/patience 20 | TEMP_IDX=0


In [ ]:
# === 5) Load all 3 data sources ===
import numpy as np
torch.manual_seed(SEED); np.random.seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"

print("loading train (20k, compressed)..."); TR = load_ds_compressed(TRAIN_DIR, DEV, verbose=True)
print("loading simulated test (existing, unchanged)..."); TE_SIM = load_ds_existing(SIM_TEST_DIR, DEV, verbose=True, exo_stats=TR["exo_stats"])
print("loading real test..."); TE_REAL = load_real_set(REAL_TEST_PATH)

print(f"\nTRAIN  N={TR['N']}  kw_ok={TR['kw_ok']}  n_time={TR['Ft']} n_weather={TR['Fw']}")
print(f"SIM TEST  N={TE_SIM['N']}  kw_ok={TE_SIM['kw_ok']}")
print(f"REAL TEST  N={len(TE_REAL)}  com={sum(1 for b in TE_REAL if b['building_type']>0)}  res={sum(1 for b in TE_REAL if b['building_type']<0)}")
assert TR["kw_ok"] and TE_SIM["kw_ok"], "kW inversion failed somewhere -- check transformers.pkl"

loading train (20k, compressed)...


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


  [transform] 0.005% overflow -> per-building median
  [transform] inverted to kW
loading simulated test (existing, unchanged)...


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PowerTransformer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  [transform] inverted to kW (type=dict)
loading real test...

TRAIN  N=20089  kw_ok=True  n_time=6 n_weather=7
SIM TEST  N=1480  kw_ok=True
REAL TEST  N=1923  com=970  res=953


In [ ]:
# === 6) Train all models x both conditions (200ep/patience20/cosine LR), RESUMABLE.
#     Zero-parameter models (persistence_ensemble) are closed-form -- skip training, matching
#     how the paper itself treats persistence baselines ("Not pretrained + Not fine-tuned").
#     Per-building NRMSE/NMAE/CRPS arrays are saved alongside the summary CSV -- needed for the
#     bootstrap CIs and significance tests in the results cell (a point estimate alone can't
#     support a "beats X" claim). ===
import csv, time, pandas as pd, pickle, traceback
if RESET and os.path.exists(SIM_CSV): os.remove(SIM_CSV); print("RESET: cleared old sim results")
FIELDS=["model","condition","weather","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","val_nll","params","sec"]
if not os.path.exists(SIM_CSV): csv.DictWriter(open(SIM_CSV,"w",newline=""),FIELDS).writeheader()
done=set()
if os.path.getsize(SIM_CSV)>50:
    d=pd.read_csv(SIM_CSV); done={(r.model,r.condition) for _,r in d.iterrows()}
PERBUILDING_DIR=f"{RESULTS_DIR}/perbuilding_sim"; os.makedirs(PERBUILDING_DIR,exist_ok=True)

run_list=[(m,c) for m in ALL_MODELS for c in CONDITIONS]
print(f"{len(run_list)} runs ({len(ALL_MODELS)} models x {len(CONDITIONS)} conditions)\n")
for name,cond in run_list:
    if (name,cond) in done: print(f"skip {name}/{cond}"); continue
    use_w=CONDITIONS[cond]; t0=time.time()
    try:
        if name=="xgboost":
            ckpt=f"{WEIGHTS_DIR}/xgboost_{cond}.pkl"
            if os.path.exists(ckpt):
                print(f"found existing xgboost/{cond} checkpoint -- reusing, skipping refit", flush=True)
                mdl,sig=pickle.load(open(ckpt,"rb")); params=0
            else:
                print(f"fit xgboost/{cond}...",flush=True)
                xdev="cuda" if DEV=="cuda" else "cpu"
                mdl,sig=xgb_fit_sim(gather, TR, DEV, use_w, L, H, n_windows=XGB_WINDOWS, device=xdev,
                                     n_estimators=XGB_TREES, max_depth=XGB_DEPTH)
                pickle.dump((mdl,sig),open(ckpt,"wb")); params=0   # save BEFORE eval
            res=xgb_eval_sim(mdl,sig,gather,TE_SIM,DEV,use_w,L,H,TE_SIM["kw_ok"]); val=float("nan")
        else:
            amp_this = USE_AMP and (name not in RNN_SET)
            torch.manual_seed(SEED)
            base=build(name, L=L, H=H, n_time=TR["Ft"], n_weather=TR["Fw"], use_weather=use_w,
                       **MODEL_KW[name]).to(DEV)
            params=count_params(base)
            ckpt=f"{WEIGHTS_DIR}/{name}_{cond}.pt"
            if params>0 and os.path.exists(ckpt):
                print(f"found existing {name}/{cond} checkpoint -- reusing, skipping retrain", flush=True)
                base.load_state_dict(torch.load(ckpt, map_location=DEV)); val=float("nan")
            elif params==0:
                print(f"{name}/{cond}: 0 params, closed-form -- skipping training", flush=True)
                val=float("nan")
            else:
                print(f"train {name}/{cond} ({params:,}p, amp={amp_this})...", flush=True)
                val=train(base, base, TR, DEV, use_w, epochs=EPOCHS, patience=PATIENCE, steps=STEPS, bs=BS, lr=LR,
                          amp=amp_this, amp_dtype=AMP_DTYPE, clip=1.0, log=True)
                torch.save(base.state_dict(), ckpt)   # save BEFORE eval -- a crash in evaluate() can no longer lose trained weights
            res=evaluate(base, TE_SIM, DEV, use_w, amp=amp_this, amp_dtype=AMP_DTYPE)
        pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res")},
                    open(f"{PERBUILDING_DIR}/{name}_{cond}.pkl","wb"))
        row={"model":name,"condition":cond,"weather":use_w,
             "com_nrmse":round(res.get("Com NRMSE",float('nan')),3),"res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
             "com_nmae":round(res.get("Com NMAE",float('nan')),3),"res_nmae":round(res.get("Res NMAE",float('nan')),3),
             "com_crps":round(res.get("Com CRPS",float('nan')),3),"res_crps":round(res.get("Res CRPS",float('nan')),3),
             "val_nll":(round(val,4) if val==val else ""),"params":params,"sec":round(time.time()-t0)}
        csv.DictWriter(open(SIM_CSV,"a",newline=""),FIELDS).writerow(row)
        print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}\n")
    except Exception as e:
        print(f"  FAILED {name}/{cond}: {type(e).__name__}: {e}"); traceback.print_exc()
print("DONE ->", SIM_CSV)

RESET: cleared old sim results
26 runs (13 models x 2 conditions)

train pwfnet/A (1,045,360p, amp=True)...
    ep000 val=0.8477 best=0.8477 bad=0
    ep010 val=0.5478 best=0.5478 bad=0
    ep020 val=0.5070 best=0.5070 bad=0
    ep030 val=0.4590 best=0.4590 bad=0
    ep040 val=0.4223 best=0.4183 bad=3
    ep050 val=0.4195 best=0.3910 bad=7
    ep060 val=0.3662 best=0.3662 bad=0
    ep070 val=0.3712 best=0.3662 bad=10
    ep080 val=0.3785 best=0.3562 bad=5
    ep090 val=0.3905 best=0.3538 bad=3
    ep100 val=0.3396 best=0.3218 bad=2
    ep110 val=0.3563 best=0.3218 bad=12
    early stop @ep118
  done 256s  Com NRMSE=17.744  Res NRMSE=43.016

train pwfnet/B (1,045,619p, amp=True)...
    ep000 val=0.8408 best=0.8408 bad=0
    ep010 val=0.5625 best=0.5285 bad=1
    ep020 val=0.4888 best=0.4872 bad=2
    ep030 val=0.4121 best=0.4121 bad=0
    ep040 val=0.4099 best=0.4028 bad=2
    ep050 val=0.4347 best=0.3790 bad=1
    ep060 val=0.3633 best=0.3633 bad=0
    ep070 val=0.3656 best=0.3618 bad=

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [18:08:52] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  done 28s  Com NRMSE=20.395  Res NRMSE=44.681

fit xgboost/B...
  done 38s  Com NRMSE=18.82  Res NRMSE=41.34

DONE -> /content/drive/MyDrive/quick/bench/novel20k_results/sim_results.csv


In [ ]:
# === 7) Real-eval: no-weather checkpoints only. Saves per-building arrays for CI/significance too. ===
import csv, time, pandas as pd, pickle, traceback, numpy as np
if RESET and os.path.exists(REAL_CSV): os.remove(REAL_CSV); print("RESET: cleared old real results")
RFIELDS=["model","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","n_buildings","sec"]
if not os.path.exists(REAL_CSV): csv.DictWriter(open(REAL_CSV,"w",newline=""),RFIELDS).writeheader()
rdone=set()
if os.path.getsize(REAL_CSV)>50:
    d=pd.read_csv(REAL_CSV); rdone={r.model for _,r in d.iterrows()}
PERBUILDING_REAL_DIR=f"{RESULTS_DIR}/perbuilding_real"; os.makedirs(PERBUILDING_REAL_DIR,exist_ok=True)

for name in ALL_MODELS:
    if name in rdone: print(f"skip {name} (real-eval)"); continue
    t0=time.time()
    try:
        if name=="xgboost":
            mdl,sig=pickle.load(open(f"{WEIGHTS_DIR}/xgboost_A.pkl","rb"))
            se=np.zeros(len(TE_REAL)); sae=np.zeros(len(TE_REAL)); sy=np.zeros(len(TE_REAL)); cnt=np.zeros(len(TE_REAL))
            for bi,b in enumerate(TE_REAL):
                ser=b["series"].values; n=len(ser)
                if n<WIN: continue
                starts=list(range(0,n-WIN,24))[:90]
                for s0 in starts:
                    win=ser[s0:s0+WIN]; yh_=win[:L]; yf_=win[L:]
                    mu_=yh_.mean(); sd_=max(yh_.std(),1e-3)
                    cal=calendar_features(b["series"].index[s0:s0+WIN])
                    btype_col=np.full((WIN,1), b["building_type"], dtype=np.float32)
                    exo=np.concatenate([cal, btype_col],1)
                    x=np.concatenate([(yh_-mu_)/sd_, exo[L:].reshape(-1)])
                    pred=mdl.predict(x.reshape(1,-1))[0]*sd_+mu_
                    se[bi]+=((pred-yf_)**2).sum(); sae[bi]+=np.abs(pred-yf_).sum(); sy[bi]+=yf_.sum(); cnt[bi]+=H
            valid=cnt>0; meany=np.zeros(len(TE_REAL)); meany[valid]=sy[valid]/cnt[valid]
            ok=valid&(meany>1e-6)
            nrmse=np.full(len(TE_REAL),np.nan); nmae=np.full(len(TE_REAL),np.nan)
            nrmse[ok]=100*np.sqrt(se[ok]/cnt[ok])/meany[ok]; nmae[ok]=100*(sae[ok]/cnt[ok])/meany[ok]
            bt=np.array([b["building_type"] for b in TE_REAL])
            res={"n_buildings":int(ok.sum()), "_nrmse":nrmse,"_nmae":nmae,
                 "_crps":np.full(len(TE_REAL),np.nan),"_is_res":(bt<0),"_valid":ok}
            for lab,mask in [("Com",(bt>0)&ok),("Res",(bt<0)&ok)]:
                if mask.any():
                    res[f"{lab} NRMSE"]=float(np.nanmedian(nrmse[mask])); res[f"{lab} NMAE"]=float(np.nanmedian(nmae[mask]))
        else:
            base=build(name, L=L, H=H, n_time=TR["Ft"], n_weather=TR["Fw"], use_weather=False,
                       **MODEL_KW[name]).to(DEV)
            sd_path=f"{WEIGHTS_DIR}/{name}_A.pt"
            if count_params(base)>0:
                if not os.path.exists(sd_path): print(f"  {name}: no no-weather checkpoint yet, skip"); continue
                base.load_state_dict(torch.load(sd_path, map_location=DEV))
            print(f"real-eval {name}...", flush=True)
            res=evaluate_real(base, TE_REAL, DEV, stride=24, max_windows_per_building=90,
                               amp=USE_AMP and name not in RNN_SET, amp_dtype=AMP_DTYPE)
        pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res") if k in res},
                    open(f"{PERBUILDING_REAL_DIR}/{name}.pkl","wb"))
        row={"model":name,"com_nrmse":round(res.get("Com NRMSE",float('nan')),3),
             "res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
             "com_nmae":round(res.get("Com NMAE",float('nan')),3) if res.get("Com NMAE",float('nan'))==res.get("Com NMAE",float('nan')) else "",
             "res_nmae":round(res.get("Res NMAE",float('nan')),3) if res.get("Res NMAE",float('nan'))==res.get("Res NMAE",float('nan')) else "",
             "com_crps":round(res.get("Com CRPS",float('nan')),3) if res.get("Com CRPS",float('nan'))==res.get("Com CRPS",float('nan')) else "",
             "res_crps":round(res.get("Res CRPS",float('nan')),3) if res.get("Res CRPS",float('nan'))==res.get("Res CRPS",float('nan')) else "",
             "n_buildings":res.get("n_buildings",0),"sec":round(time.time()-t0)}
        csv.DictWriter(open(REAL_CSV,"a",newline=""),RFIELDS).writerow(row)
        print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}  (n={row['n_buildings']})\n")
    except Exception as e:
        print(f"  FAILED {name}: {type(e).__name__}: {e}"); traceback.print_exc()
print("DONE ->", REAL_CSV)

RESET: cleared old real results
real-eval pwfnet...
  done 43s  Com NRMSE=12.956  Res NRMSE=75.059  (n=1922)

real-eval xpan...
  done 36s  Com NRMSE=14.256  Res NRMSE=76.66  (n=1922)

real-eval freqmoe...
  done 35s  Com NRMSE=12.928  Res NRMSE=75.91  (n=1922)

real-eval drdt...
  done 35s  Com NRMSE=12.877  Res NRMSE=75.062  (n=1922)

real-eval mscah...
  done 36s  Com NRMSE=13.258  Res NRMSE=76.321  (n=1922)

real-eval naive...
  done 35s  Com NRMSE=16.433  Res NRMSE=98.457  (n=1922)

real-eval linreg...
  done 34s  Com NRMSE=13.33  Res NRMSE=79.506  (n=1922)

real-eval lstm...
  done 35s  Com NRMSE=14.44  Res NRMSE=77.045  (n=1922)

real-eval gru...
  done 35s  Com NRMSE=14.466  Res NRMSE=77.042  (n=1922)

real-eval cnn_lstm...
  done 39s  Com NRMSE=14.631  Res NRMSE=76.679  (n=1922)

real-eval dlinear...
  done 34s  Com NRMSE=13.483  Res NRMSE=80.058  (n=1922)

real-eval persistence_ensemble...
  done 35s  Com NRMSE=16.178  Res NRMSE=76.879  (n=1922)

  done 1005s  Com NRMSE=12.74

In [ ]:
# === 8) Results: median [95% bootstrap CI], NMAE, CRPS, and significance vs Persistence Ensemble
#     (paired Wilcoxon signed-rank test on matched per-building NRMSE -- a numerically lower
#     median alone doesn't support a "beats X" claim without this). ===
import pandas as pd, pickle, numpy as np
order = list(MODEL_KW.keys()) + ["xgboost"]
BASELINE = "persistence_ensemble"   # paper's own reference point for "does this beat naive persistence"

def _load_pb(path):
    try: return pickle.load(open(path,"rb"))
    except FileNotFoundError: return None

def _row(name, pb, mask_key, baseline_pb):
    if pb is None: return None
    mask = pb["_is_res"] if mask_key=="Res" else ~pb["_is_res"]
    vals = pb["_nrmse"][mask]
    med, lo, hi = bootstrap_median_ci(vals)
    sig = ""
    if baseline_pb is not None and name != BASELINE:
        bvals = baseline_pb["_nrmse"][mask]
        p, better = paired_significance(vals, bvals)
        if better is True: sig = f" *p={p:.1e}, beats {BASELINE}*"
        elif better is False and p==p and p<0.05: sig = f" (p={p:.1e}, WORSE than {BASELINE})"
    return med, lo, hi, sig

print("=== SIMULATED (Buildings-900K-Test) === paper best: Com 12.91 (900K-pretrained zero-shot) | prior round's TimesNet/B: 12.07 (10k)")
print(f"Significance: paired Wilcoxon signed-rank test on matched per-building NRMSE vs {BASELINE}, alpha=0.05\n")
for cond in ["A","B"]:
    print(f"-- Condition {cond} ({'no weather' if cond=='A' else '+weather'}) --")
    base_pb = _load_pb(f"{PERBUILDING_DIR}/{BASELINE}_{cond}.pkl")
    for name in order:
        pb = _load_pb(f"{PERBUILDING_DIR}/{name}_{cond}.pkl")
        r = _row(name, pb, "Com", base_pb)
        if r is None: continue
        med,lo,hi,sig = r
        print(f"  {name:20s} Com {med:6.2f} [{lo:5.2f},{hi:5.2f}]{sig}")
    print()

print(f"=== REAL (BuildingsBench, 7 datasets, no-weather) === paper best: Com 13.31 | Res 79.34 (900K-pretrained zero-shot)\n")
base_pb_r = _load_pb(f"{PERBUILDING_REAL_DIR}/{BASELINE}.pkl")
for name in order:
    pb = _load_pb(f"{PERBUILDING_REAL_DIR}/{name}.pkl")
    r = _row(name, pb, "Com", base_pb_r)
    if r is None: continue
    med,lo,hi,sig = r
    print(f"  {name:20s} Com {med:6.2f} [{lo:5.2f},{hi:5.2f}]{sig}")

In [ ]:
# === 8) Results: median [95% bootstrap CI], NMAE, CRPS, and significance vs Persistence Ensemble
#     (paired Wilcoxon signed-rank test on matched per-building NRMSE -- a numerically lower
#     median alone doesn't support a "beats X" claim without this). ===
import pandas as pd, pickle, numpy as np
order = list(MODEL_KW.keys()) + ["xgboost"]
BASELINE = "persistence_ensemble"   # paper's own reference point for "does this beat naive persistence"
if os.path.getsize(SIM_CSV)>50:
    d=pd.read_csv(SIM_CSV); done={(r.model,r.condition) for _,r in d.iterrows()}
PERBUILDING_DIR=f"{RESULTS_DIR}/perbuilding_sim"; os.makedirs(PERBUILDING_DIR,exist_ok=True)
def _load_pb(path):
    try: return pickle.load(open(path,"rb"))
    except FileNotFoundError: return None

def _row(name, pb, mask_key, baseline_pb):
    if pb is None: return None
    mask = pb["_is_res"] if mask_key=="Res" else ~pb["_is_res"]
    vals = pb["_nrmse"][mask]
    med, lo, hi = bootstrap_median_ci(vals)
    sig = ""
    if baseline_pb is not None and name != BASELINE:
        bvals = baseline_pb["_nrmse"][mask]
        p, better = paired_significance(vals, bvals)
        if better is True: sig = f" *p={p:.1e}, beats {BASELINE}*"
        elif better is False and p==p and p<0.05: sig = f" (p={p:.1e}, WORSE than {BASELINE})"
    return med, lo, hi, sig

print("=== SIMULATED (Buildings-900K-Test) === paper best: Com 12.91 (900K-pretrained zero-shot) | prior round's TimesNet/B: 12.07 (10k)")
print(f"Significance: paired Wilcoxon signed-rank test on matched per-building NRMSE vs {BASELINE}, alpha=0.05\n")
for cond in ["A","B"]:
    print(f"-- Condition {cond} ({'no weather' if cond=='A' else '+weather'}) --")
    base_pb = _load_pb(f"{PERBUILDING_DIR}/{BASELINE}_{cond}.pkl")
    for name in order:
        pb = _load_pb(f"{PERBUILDING_DIR}/{name}_{cond}.pkl")
        r = _row(name, pb, "Com", base_pb)
        if r is None: continue
        med,lo,hi,sig = r
        print(f"  {name:20s} Com {med:6.2f} [{lo:5.2f},{hi:5.2f}]{sig}")
    print()

print(f"=== REAL (BuildingsBench, 7 datasets, no-weather) === paper best: Com 13.31 | Res 79.34 (900K-pretrained zero-shot)\n")
base_pb_r = _load_pb(f"{PERBUILDING_REAL_DIR}/{BASELINE}.pkl")
for name in order:
    pb = _load_pb(f"{PERBUILDING_REAL_DIR}/{name}.pkl")
    r = _row(name, pb, "Com", base_pb_r)
    if r is None: continue
    med,lo,hi,sig = r
    print(f"  {name:20s} Com {med:6.2f} [{lo:5.2f},{hi:5.2f}]{sig}")

=== SIMULATED (Buildings-900K-Test) === paper best: Com 12.91 (900K-pretrained zero-shot) | prior round's TimesNet/B: 12.07 (10k)
Significance: paired Wilcoxon signed-rank test on matched per-building NRMSE vs persistence_ensemble, alpha=0.05

-- Condition A (no weather) --
  pwfnet               Com  17.74 [16.82,18.44] *p=3.1e-94, beats persistence_ensemble*
  xpan                 Com  24.40 [23.55,25.10] *p=4.7e-91, beats persistence_ensemble*
  freqmoe              Com  19.70 [18.78,20.74] *p=3.1e-94, beats persistence_ensemble*
  drdt                 Com  19.12 [18.17,20.05] *p=3.1e-94, beats persistence_ensemble*
  mscah                Com  20.25 [19.51,21.70] *p=3.3e-94, beats persistence_ensemble*
  naive                Com  34.83 [33.56,36.45] (p=2.0e-57, WORSE than persistence_ensemble)
  linreg               Com  23.59 [22.85,24.31] *p=4.3e-94, beats persistence_ensemble*
  lstm                 Com  21.32 [20.25,22.77] *p=2.1e-89, beats persistence_ensemble*
  gru           

NameError: name 'PERBUILDING_REAL_DIR' is not defined

## Optional section — does +weather actually help on real buildings?

Run this **after** cells 1–8 above have completed at least once (it needs the `_B.pt` +weather checkpoints they save). Real weather only has temperature + humidity — the other 5 channels the model expects are filled with the training-set average, not real data.

In [ ]:
# === 9) OPTIONAL SEPARATE SECTION -- run this only after cells 1-8 have completed at least once.
#     Real-weather eval: does +weather actually help on real buildings? Real weather has only 2
#     channels (temp, humidity, from era5); the other 5 the model expects are filled with 0 in
#     normalized space (= training-set average -- an approximation, not observed wind/solar). ===
# === Real-weather eval (SEPARATE, optional section): +weather checkpoints on real buildings.
#     Real weather only has 2 channels (temperature, humidity) vs the 7 the model expects
#     (temp, humidity, wind speed, wind dir, GHI, DNI, DHI -- WEATHER_COLS order, confirmed from
#     prepare_buildingsbench.py: index 0=temp, 1=humidity). The missing 5 are filled with 0 in
#     NORMALIZED space, which -- since weather is z-scored using TRAIN statistics -- literally
#     means "assume the training-set average" for wind/solar. Real, not hidden, approximation. ===
import os, subprocess, numpy as np, pandas as pd, torch, torch.nn.functional as F, math

S3_WBASE="s3://oedi-data-lake/buildings-bench/v2.0.0/BuildingsBench"
BDG2_SITES = ["Bear","Fox","Panther","Rat"]

def fetch_all_real_weather(datasets, tmp_dir):
    """Returns {dataset_name: {"default": df} or {site: df for site in BDG2_SITES}}.
    df columns: temperature (deg C), humidity (%), indexed by DatetimeIndex."""
    os.makedirs(tmp_dir, exist_ok=True); out={}
    for name in datasets:
        if name=="BDG-2":
            out[name]={}
            for site in BDG2_SITES:
                local=f"{tmp_dir}/w_{name}_{site}.csv"
                if not os.path.exists(local):
                    subprocess.run(["aws","s3","cp",f"{S3_WBASE}/{name}/weather_{site}_era5.csv",local,
                                     "--no-sign-request"],capture_output=True)
                df=pd.read_csv(local); tc=[c for c in df.columns if c.lower() in ("time","timestamp")][0]
                out[name][site]=df.set_index(pd.to_datetime(df[tc]))[["temperature","humidity"]].sort_index()
        else:
            local=f"{tmp_dir}/w_{name}.csv"
            if not os.path.exists(local):
                subprocess.run(["aws","s3","cp",f"{S3_WBASE}/{name}/weather_era5.csv",local,
                                 "--no-sign-request"],capture_output=True)
            df=pd.read_csv(local); tc=[c for c in df.columns if c.lower() in ("time","timestamp")][0]
            out[name]={"default": df.set_index(pd.to_datetime(df[tc]))[["temperature","humidity"]].sort_index()}
    return out

def _weather_for_building(building_id, wcache):
    dataset, bid = building_id.split(":", 1)
    if dataset=="BDG-2":
        site = bid.split("_")[0]                  # column names are "{Site}_{type}_{name}"
        return wcache[dataset].get(site)
    return wcache.get(dataset, {}).get("default")

def calendar_features(index):
    hour = index.hour.values.astype(np.float32)
    dow  = index.dayofweek.values.astype(np.float32)
    doy  = index.dayofyear.values.astype(np.float32)
    return np.stack([np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24),
                      np.sin(2*np.pi*dow/7),  np.cos(2*np.pi*dow/7),
                      np.sin(2*np.pi*doy/365.25), np.cos(2*np.pi*doy/365.25)], 1).astype(np.float32)
@torch.no_grad()
def evaluate_real_weather(model, buildings, wcache, exo_stats, dev, n_time=6, n_weather=7,
                           stride=24, max_windows_per_building=90, batch=2048, amp=True, amp_dtype=torch.bfloat16):
    """Same structure as evaluate_real(), but builds a full n_weather-width exo block per window:
    channel 0=temperature, 1=humidity (real, normalized with TRAIN stats), channels 2-6=0.0
    (=training-set average, since those channels aren't observable in real data)."""

    wm, ws, tm, ts = exo_stats     # wm,ws: (n_weather,) TRAIN weather mean/std; tm,ts: (n_time,) TRAIN calendar mean/std
    model.eval(); ac = lambda: torch.autocast("cuda", dtype=amp_dtype, enabled=(amp and dev=="cuda"))
    jobs=[]; L,H,WIN=168,24,192
    for bi, b in enumerate(buildings):
        w = _weather_for_building(b["building_id"], wcache)
        if w is None: continue
        n=len(b["series"])
        if n < WIN: continue
        starts=list(range(0, n-WIN, stride))
        if len(starts) > max_windows_per_building:
            rng=np.random.default_rng(bi); starts=sorted(rng.choice(starts, max_windows_per_building, replace=False).tolist())
        jobs += [(bi,s) for s in starts]
    if not jobs: return {"n_buildings": 0, "n_weather_matched": 0}

    se=np.zeros(len(buildings)); sae=np.zeros(len(buildings)); sy=np.zeros(len(buildings))
    sy2=np.zeros(len(buildings)); cnt=np.zeros(len(buildings)); cr=np.zeros(len(buildings))
    def ncdf(x): return 0.5*(1+torch.erf(x/math.sqrt(2)))
    def npdf(x): return torch.exp(-0.5*x*x)/math.sqrt(2*math.pi)

    for i in range(0, len(jobs), batch):
        chunk=jobs[i:i+batch]
        yh_list, exo_list, yf_list, bidx = [], [], [], []
        for bi, s in chunk:
            b=buildings[bi]; ser=b["series"]; w=_weather_for_building(b["building_id"], wcache)
            win=ser.values[s:s+WIN].astype(np.float32); idx=ser.index[s:s+WIN]
            cal_raw = calendar_features(idx)                                    # (WIN,6) raw, un-normalized
            cal_n = (cal_raw - tm) / ts                                          # normalize with TRAIN calendar stats
            wsub = w.reindex(idx, method="nearest", tolerance=pd.Timedelta("2h")).interpolate().bfill().ffill()
            temp = wsub["temperature"].values.astype(np.float32)
            hum  = wsub["humidity"].values.astype(np.float32)
            wblock = np.zeros((WIN, n_weather), dtype=np.float32)
            wblock[:,0] = (temp - wm[0]) / ws[0]                                 # channel 0 = temperature (normalized)
            wblock[:,1] = (hum  - wm[1]) / ws[1]                                 # channel 1 = humidity (normalized)
            # channels 2..6 (wind speed, wind dir, GHI, DNI, DHI) stay 0.0 = training-set average
            btype_col=np.full((WIN,1), b["building_type"], dtype=np.float32)
            exo=np.concatenate([cal_n, wblock, btype_col],1)                     # (WIN, n_time+n_weather+1)
            yh_list.append(win[:L]); yf_list.append(win[L:]); exo_list.append(exo); bidx.append(bi)
        yh=torch.tensor(np.stack(yh_list),device=dev); yf=torch.tensor(np.stack(yf_list),device=dev)
        exo=torch.tensor(np.stack(exo_list),device=dev)
        with ac(): mu_n,rs,mean,sd=model(yh,exo)
        mu_n,rs,mean,sd=mu_n.float(),rs.float(),mean.float(),sd.float()
        mu=mu_n*sd+mean; sigma=(F.softplus(rs)+1e-3)*sd
        err=((mu-yf)**2).sum(1).cpu().numpy(); aerr=(mu-yf).abs().sum(1).cpu().numpy()
        ysum=yf.sum(1).cpu().numpy(); y2sum=(yf**2).sum(1).cpu().numpy()
        z=(yf-mu)/sigma
        crps=(sigma*(z*(2*ncdf(z)-1)+2*npdf(z)-1/math.sqrt(math.pi))).sum(1).cpu().numpy()
        for j,bi in enumerate(bidx):
            se[bi]+=err[j]; sae[bi]+=aerr[j]; sy[bi]+=ysum[j]; sy2[bi]+=y2sum[j]; cnt[bi]+=H; cr[bi]+=crps[j]

    valid=cnt>0; rmse=np.zeros_like(se); mae=np.zeros_like(sae); meany=np.zeros_like(sy); crps_avg=np.zeros_like(cr)
    rmse[valid]=np.sqrt(se[valid]/cnt[valid]); mae[valid]=sae[valid]/cnt[valid]
    meany[valid]=sy[valid]/cnt[valid]; crps_avg[valid]=cr[valid]/cnt[valid]
    ok=valid & (meany>1e-6)
    nrmse=np.full(len(buildings),np.nan); nmae=np.full(len(buildings),np.nan)
    nrmse[ok]=100*rmse[ok]/meany[ok]; nmae[ok]=100*mae[ok]/meany[ok]
    btypes=np.array([b["building_type"] for b in buildings])
    out={"n_buildings": int(ok.sum()), "n_weather_matched": len(set(bi for bi,_ in jobs))}
    for lab, mask in [("Com", (btypes>0)&ok), ("Res", (btypes<0)&ok)]:
        if mask.any():
            out[f"{lab} NRMSE"]=float(np.nanmedian(nrmse[mask])); out[f"{lab} NMAE"]=float(np.nanmedian(nmae[mask]))
            out[f"{lab} CRPS"]=float(np.median(crps_avg[mask]))
    out["_nrmse"]=nrmse; out["_nmae"]=nmae; out["_crps"]=crps_avg; out["_is_res"]=(btypes<0); out["_valid"]=ok
    return out
print("real-weather eval ready")

real-weather eval ready


In [ ]:
!pip install -q awscli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 147.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


In [ ]:
# === 10) Fetch real weather (era5, all 7 datasets) + evaluate +weather checkpoints on real buildings ===
import time, pickle
t0=time.time()
WCACHE = fetch_all_real_weather(REAL_DATASETS if "REAL_DATASETS" in dir() else
                                 sorted(set(b["building_id"].split(":")[0] for b in TE_REAL)),
                                 "/content/dl_realweather")
print(f"real weather fetched ({time.time()-t0:.0f}s)\n")

RW_CSV=f"{RESULTS_DIR}/real_weather_results.csv"
PERBUILDING_RW_DIR=f"{RESULTS_DIR}/perbuilding_real_weather"; os.makedirs(PERBUILDING_RW_DIR,exist_ok=True)
import csv, pandas as pd, traceback
RWFIELDS=["model","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","n_buildings","n_weather_matched","sec"]
if RESET and os.path.exists(RW_CSV): os.remove(RW_CSV)
if not os.path.exists(RW_CSV): csv.DictWriter(open(RW_CSV,"w",newline=""),RWFIELDS).writeheader()
rwdone=set()
if os.path.getsize(RW_CSV)>50:
    d=pd.read_csv(RW_CSV); rwdone={r.model for _,r in d.iterrows()}

for name in [m for m in ALL_MODELS if m!="xgboost"]:   # xgboost's feature layout is condition-specific; skip here
    if name in rwdone: print(f"skip {name} (real-weather eval)"); continue
    t0=time.time()
    try:
        base=build(name, L=L, H=H, n_time=TR["Ft"], n_weather=TR["Fw"], use_weather=True, **MODEL_KW[name]).to(DEV)
        if count_params(base)>0:
            sd_path=f"{WEIGHTS_DIR}/{name}_B.pt"
            if not os.path.exists(sd_path): print(f"  {name}: no +weather checkpoint yet, skip"); continue
            base.load_state_dict(torch.load(sd_path, map_location=DEV))
        print(f"real-weather-eval {name}...", flush=True)
        res=evaluate_real_weather(base, TE_REAL, WCACHE, TR["exo_stats"], DEV, n_time=TR["Ft"], n_weather=TR["Fw"],
                                   amp=USE_AMP and name not in RNN_SET, amp_dtype=AMP_DTYPE)
        pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res") if k in res},
                    open(f"{PERBUILDING_RW_DIR}/{name}.pkl","wb"))
        row={"model":name,"com_nrmse":round(res.get("Com NRMSE",float('nan')),3),
             "res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
             "com_nmae":round(res.get("Com NMAE",float('nan')),3) if res.get("Com NMAE",float('nan'))==res.get("Com NMAE",float('nan')) else "",
             "res_nmae":round(res.get("Res NMAE",float('nan')),3) if res.get("Res NMAE",float('nan'))==res.get("Res NMAE",float('nan')) else "",
             "com_crps":round(res.get("Com CRPS",float('nan')),3) if res.get("Com CRPS",float('nan'))==res.get("Com CRPS",float('nan')) else "",
             "res_crps":round(res.get("Res CRPS",float('nan')),3) if res.get("Res CRPS",float('nan'))==res.get("Res CRPS",float('nan')) else "",
             "n_buildings":res.get("n_buildings",0),"n_weather_matched":res.get("n_weather_matched",0),
             "sec":round(time.time()-t0)}
        csv.DictWriter(open(RW_CSV,"a",newline=""),RWFIELDS).writerow(row)
        print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}  (matched={row['n_weather_matched']})\n")
    except Exception as e:
        print(f"  FAILED {name}: {type(e).__name__}: {e}"); traceback.print_exc()
print("DONE ->", RW_CSV)

real weather fetched (0s)

real-weather-eval pwfnet...
  done 335s  Com NRMSE=12.946  Res NRMSE=75.181  (matched=1923)

real-weather-eval xpan...
  done 335s  Com NRMSE=14.584  Res NRMSE=78.356  (matched=1923)

real-weather-eval freqmoe...
  done 335s  Com NRMSE=12.855  Res NRMSE=75.788  (matched=1923)

real-weather-eval drdt...
  done 335s  Com NRMSE=12.86  Res NRMSE=75.213  (matched=1923)

real-weather-eval mscah...
  done 334s  Com NRMSE=13.432  Res NRMSE=77.891  (matched=1923)

real-weather-eval naive...
  done 333s  Com NRMSE=16.433  Res NRMSE=98.457  (matched=1923)

real-weather-eval linreg...
  done 333s  Com NRMSE=13.541  Res NRMSE=81.411  (matched=1923)

real-weather-eval lstm...
  done 333s  Com NRMSE=14.024  Res NRMSE=80.247  (matched=1923)

real-weather-eval gru...
  done 334s  Com NRMSE=13.664  Res NRMSE=82.547  (matched=1923)

real-weather-eval cnn_lstm...
  done 333s  Com NRMSE=13.806  Res NRMSE=80.899  (matched=1923)

real-weather-eval dlinear...
  done 333s  Com NRMSE=

In [ ]:
# === 10) Fetch real weather (era5, all 7 datasets) + evaluate +weather checkpoints on real buildings ===
import time, pickle
t0=time.time()
WCACHE = fetch_all_real_weather(REAL_DATASETS if "REAL_DATASETS" in dir() else
                                 sorted(set(b["building_id"].split(":")[0] for b in TE_REAL)),
                                 "/content/dl_realweather")
print(f"real weather fetched ({time.time()-t0:.0f}s)\n")

RW_CSV=f"{RESULTS_DIR}/real_weather_results.csv"
PERBUILDING_RW_DIR=f"{RESULTS_DIR}/perbuilding_real_weather"; os.makedirs(PERBUILDING_RW_DIR,exist_ok=True)
import csv, pandas as pd, traceback
RWFIELDS=["model","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","n_buildings","n_weather_matched","sec"]
if RESET and os.path.exists(RW_CSV): os.remove(RW_CSV)
if not os.path.exists(RW_CSV): csv.DictWriter(open(RW_CSV,"w",newline=""),RWFIELDS).writeheader()
rwdone=set()
if os.path.getsize(RW_CSV)>50:
    d=pd.read_csv(RW_CSV); rwdone={r.model for _,r in d.iterrows()}

for name in [m for m in ALL_MODELS if m!="xgboost"]:   # xgboost's feature layout is condition-specific; skip here
    if name in rwdone: print(f"skip {name} (real-weather eval)"); continue
    t0=time.time()
    try:
        base=build(name, L=L, H=H, n_time=TR["Ft"], n_weather=TR["Fw"], use_weather=True, **MODEL_KW[name]).to(DEV)
        if count_params(base)>0:
            sd_path=f"{WEIGHTS_DIR}/{name}_B.pt"
            if not os.path.exists(sd_path): print(f"  {name}: no +weather checkpoint yet, skip"); continue
            base.load_state_dict(torch.load(sd_path, map_location=DEV))
        print(f"real-weather-eval {name}...", flush=True)
        res=evaluate_real_weather(base, TE_REAL, WCACHE, TR["exo_stats"], DEV, n_time=TR["Ft"], n_weather=TR["Fw"],
                                   amp=USE_AMP and name not in RNN_SET, amp_dtype=AMP_DTYPE)
        pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res") if k in res},
                    open(f"{PERBUILDING_RW_DIR}/{name}.pkl","wb"))
        row={"model":name,"com_nrmse":round(res.get("Com NRMSE",float('nan')),3),
             "res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
             "com_nmae":round(res.get("Com NMAE",float('nan')),3) if res.get("Com NMAE",float('nan'))==res.get("Com NMAE",float('nan')) else "",
             "res_nmae":round(res.get("Res NMAE",float('nan')),3) if res.get("Res NMAE",float('nan'))==res.get("Res NMAE",float('nan')) else "",
             "com_crps":round(res.get("Com CRPS",float('nan')),3) if res.get("Com CRPS",float('nan'))==res.get("Com CRPS",float('nan')) else "",
             "res_crps":round(res.get("Res CRPS",float('nan')),3) if res.get("Res CRPS",float('nan'))==res.get("Res CRPS",float('nan')) else "",
             "n_buildings":res.get("n_buildings",0),"n_weather_matched":res.get("n_weather_matched",0),
             "sec":round(time.time()-t0)}
        csv.DictWriter(open(RW_CSV,"a",newline=""),RWFIELDS).writerow(row)
        print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}  (matched={row['n_weather_matched']})\n")
    except Exception as e:
        print(f"  FAILED {name}: {type(e).__name__}: {e}"); traceback.print_exc()
print("DONE ->", RW_CSV)

KeyboardInterrupt: 

In [ ]:
# === 11) Real-weather results, side-by-side with the no-weather real results from cell 7 ===
import pandas as pd
real_nw = pd.read_csv(REAL_CSV).set_index("model")
real_w  = pd.read_csv(RW_CSV).set_index("model")
order = list(MODEL_KW.keys())
print("Com NRMSE: no-weather (cell 7) vs +weather-with-real-approx-data (this section)\n")
print(f"{'model':20s} {'no-weather':>12s} {'+weather':>12s} {'delta':>8s}")
for name in order:
    if name in real_nw.index and name in real_w.index:
        a=real_nw.loc[name,"com_nrmse"]; b=real_w.loc[name,"com_nrmse"]
        print(f"{name:20s} {a:12.3f} {b:12.3f} {b-a:+8.3f}{'  (weather helps)' if b<a else ''}")

Com NRMSE: no-weather (cell 7) vs +weather-with-real-approx-data (this section)

model                  no-weather     +weather    delta
pwfnet                     12.956       12.946   -0.010  (weather helps)
xpan                       14.256       14.584   +0.328
freqmoe                    12.928       12.855   -0.073  (weather helps)
drdt                       12.877       12.860   -0.017  (weather helps)
mscah                      13.258       13.432   +0.174
naive                      16.433       16.433   +0.000
linreg                     13.330       13.541   +0.211
lstm                       14.440       14.024   -0.416  (weather helps)
gru                        14.466       13.664   -0.802  (weather helps)
cnn_lstm                   14.631       13.806   -0.825  (weather helps)
dlinear                    13.483       13.525   +0.042
persistence_ensemble       16.178       16.178   +0.000


In [ ]:
# =====================================================================================
# APPEND: 4 more models, built on this run's evidence + the prior TimesNet result.
#   timesnetxl          -- original TimesNet mechanism (no fusion extras), 2-3x capacity
#   neuralgboosthybrid  -- TimesNet backbone + internal boosting-style residual correction
#   robusttimesnet      -- TimesNet backbone + training-time noise/dropout (sim->real gap)
#   stackedensemble     -- learned combiner over your ALREADY-TRAINED xgboost+drdt+freqmoe+
#                          pwfnet checkpoints (frozen, no retraining -- reuses prior compute)
# Assumes cells 1-8 (and 9-10 if run) have already executed in this session.
# =====================================================================================
import torch.nn as nn, torch.nn.functional as F, math as _math

class TimesNetXL(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=160,d_ff=320,periods=(12,24,168),dropout=0.15,**kw):
        super().__init__()
        self.L,self.H=L,H; self.seq=L+H; self.revin=RevIN()
        self.embed=nn.Linear(1,d_model); self.pos=nn.Parameter(torch.randn(1,self.seq,d_model)*0.02)
        self.blocks=nn.ModuleList([TimesBlock2(d_model,d_ff,p) for p in periods])
        self.extra=nn.ModuleList([TimesBlock2(d_model,d_ff,p) for p in periods])
        self.norm=nn.LayerNorm(d_model); self.drop=nn.Dropout(dropout)
        self.head=nn.Linear(self.seq*d_model, 2*H)
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        full=torch.cat([yn,torch.zeros(B,self.H,device=yh.device,dtype=yn.dtype)],1)
        z=self.embed(full.unsqueeze(-1))+self.pos
        for blk in self.blocks: z=self.drop(self.norm(z+blk(z)))
        for blk in self.extra:  z=self.drop(self.norm(z+blk(z)))
        out=self.head(z.reshape(B,-1)).view(B,self.H,2)
        return out[...,0],out[...,1],mu,sd

class NeuralGBoostHybrid(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=96,d_ff=160,periods=(24,168),dropout=0.1,dh=96,**kw):
        super().__init__()
        self.L,self.H=L,H; self.seq=L+H; self.revin=RevIN()
        self.embed=nn.Linear(1,d_model); self.pos=nn.Parameter(torch.randn(1,self.seq,d_model)*0.02)
        self.blocks=nn.ModuleList([TimesBlock2(d_model,d_ff,p) for p in periods])
        self.norm=nn.LayerNorm(d_model)
        self.stage1=nn.Linear(self.seq*d_model, H)
        self.stage2=nn.Sequential(nn.Linear(self.seq*d_model+H, dh), nn.GELU(), nn.Linear(dh, H))
        self.scale_head=nn.Sequential(nn.Linear(self.seq*d_model, dh), nn.GELU(), nn.Linear(dh, H))
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        full=torch.cat([yn,torch.zeros(B,self.H,device=yh.device,dtype=yn.dtype)],1)
        z=self.embed(full.unsqueeze(-1))+self.pos
        for blk in self.blocks: z=self.norm(z+blk(z))
        flat=z.reshape(B,-1)
        stage1_out=self.stage1(flat)
        correction=self.stage2(torch.cat([flat, stage1_out.detach()],-1))
        return stage1_out+correction, self.scale_head(flat), mu, sd

class RobustTimesNet(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=96,d_ff=160,periods=(24,168),dropout=0.25,noise_std=0.03,**kw):
        super().__init__()
        self.L,self.H=L,H; self.seq=L+H; self.revin=RevIN(); self.noise_std=noise_std
        self.embed=nn.Linear(1,d_model); self.pos=nn.Parameter(torch.randn(1,self.seq,d_model)*0.02)
        self.blocks=nn.ModuleList([TimesBlock2(d_model,d_ff,p) for p in periods])
        self.norm=nn.LayerNorm(d_model); self.drop=nn.Dropout(dropout)
        self.head=nn.Linear(self.seq*d_model, 2*H)
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        if self.training and self.noise_std>0: yn = yn + torch.randn_like(yn)*self.noise_std
        full=torch.cat([yn,torch.zeros(B,self.H,device=yh.device,dtype=yn.dtype)],1)
        z=self.embed(full.unsqueeze(-1))+self.pos
        for blk in self.blocks: z=self.drop(self.norm(z+blk(z)))
        out=self.head(z.reshape(B,-1)).view(B,self.H,2)
        return out[...,0],out[...,1],mu,sd

def _inv_softplus(x, eps=1e-3):
    y = (x - eps).clamp_min(1e-4)
    return torch.where(y>20, y, torch.log(torch.expm1(y)))

class StackedEnsemble(nn.Module):
    """base_nets: 4 frozen already-trained models (xgboost handled separately)."""
    def __init__(self, base_nets, xgb_model, xgb_sigma, L=168, H=24, dh=64, **kw):
        super().__init__()
        self.L, self.H = L, H
        self.base_nets = nn.ModuleList(base_nets)
        for p in self.base_nets.parameters(): p.requires_grad_(False)
        self.xgb_model, self.xgb_sigma = xgb_model, xgb_sigma
        n_base = len(base_nets) + 1
        self.mu_combiner = nn.Sequential(nn.Linear(n_base*H, dh), nn.GELU(), nn.Linear(dh, H))
        self.sig_combiner = nn.Sequential(nn.Linear(n_base*H, dh), nn.GELU(), nn.Linear(dh, H))
    def forward(self, yh, exo):
        B = yh.size(0); dev = yh.device
        mean = yh.mean(1, keepdim=True); sd = yh.std(1, keepdim=True).clamp_min(1e-3)
        mus, sigs = [], []
        with torch.no_grad():
            for net in self.base_nets:
                mu_n, raw, m, s = net(yh, exo)
                mus.append(mu_n.float()*s.float()+m.float()); sigs.append((F.softplus(raw.float())+1e-3)*s.float())
            X, xm, xs = _xgb_xy(yh, exo, self.L)
            P = self.xgb_model.predict(X)
            xgb_mu = torch.tensor(P*xs[:,None]+xm[:,None], device=dev, dtype=torch.float32)
            xgb_sig = torch.tensor(self.xgb_sigma[None,:]*xs[:,None], device=dev, dtype=torch.float32).expand(B,self.H).contiguous()
            mus.append(xgb_mu); sigs.append(xgb_sig)
        mu_stack = torch.cat(mus, -1); sig_stack = torch.cat(sigs, -1)
        mu_phys = self.mu_combiner(mu_stack)
        sig_phys = F.softplus(self.sig_combiner(sig_stack)) + 1e-3
        return (mu_phys-mean)/sd, _inv_softplus(sig_phys/sd), mean, sd

V2_KW = {
  "timesnetxl":         dict(d_model=160, d_ff=320, periods=(12,24,168), dropout=0.15),
  "neuralgboosthybrid": dict(d_model=96,  d_ff=160, periods=(24,168), dropout=0.1, dh=96),
  "robusttimesnet":     dict(d_model=96,  d_ff=160, periods=(24,168), dropout=0.25, noise_std=0.03),
  "stackedensemble":    dict(dh=64),
}
V2_REG = {"timesnetxl":TimesNetXL, "neuralgboosthybrid":NeuralGBoostHybrid, "robusttimesnet":RobustTimesNet}
REG.update(V2_REG)   # so build(name,...) resolves the new names
ENSEMBLE_BASES = ["xgboost","drdt","freqmoe","pwfnet"]   # different failure modes: tree / physics-branch / gated-experts / TimesNet+physics

MODEL_KW.update(V2_KW)
ALL_MODELS = ALL_MODELS + list(V2_KW.keys())
print(f"models now: {ALL_MODELS}")

# ---- train loop: same pattern as cell 6, appends new rows only ----
import csv, time, pandas as pd, pickle, traceback
FIELDS=["model","condition","weather","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","val_nll","params","sec"]
done=set()
if os.path.getsize(SIM_CSV)>50:
    d=pd.read_csv(SIM_CSV); done={(r.model,r.condition) for _,r in d.iterrows()}
PERBUILDING_DIR=f"{RESULTS_DIR}/perbuilding_sim"; os.makedirs(PERBUILDING_DIR,exist_ok=True)

for name in list(V2_KW.keys()):
    for cond in CONDITIONS:
        if (name,cond) in done: print(f"skip {name}/{cond}"); continue
        use_w=CONDITIONS[cond]; t0=time.time()
        try:
            torch.manual_seed(SEED)
            if name=="stackedensemble":
                base_nets=[]
                for bn in ENSEMBLE_BASES:
                    if bn=="xgboost": continue
                    bm=build(bn, L=L,H=H,n_time=TR["Ft"],n_weather=TR["Fw"],use_weather=use_w,**MODEL_KW[bn]).to(DEV)
                    bm.load_state_dict(torch.load(f"{WEIGHTS_DIR}/{bn}_{cond}.pt", map_location=DEV)); bm.eval()
                    base_nets.append(bm)
                xgb_mdl,xgb_sig = pickle.load(open(f"{WEIGHTS_DIR}/xgboost_{cond}.pkl","rb"))
                base=StackedEnsemble(base_nets, xgb_mdl, xgb_sig, L=L,H=H,**V2_KW["stackedensemble"]).to(DEV)
            else:
                base=build(name, L=L,H=H,n_time=TR["Ft"],n_weather=TR["Fw"],use_weather=use_w,**MODEL_KW[name]).to(DEV)
            params=count_params(base)
            print(f"train {name}/{cond} ({params:,}p)...", flush=True)
            val=train(base, base, TR, DEV, use_w, epochs=EPOCHS, patience=PATIENCE, steps=STEPS, bs=BS, lr=LR,
                      amp=USE_AMP, amp_dtype=AMP_DTYPE, clip=1.0, log=True)
            torch.save(base.state_dict(), f"{WEIGHTS_DIR}/{name}_{cond}.pt")   # save BEFORE eval
            res=evaluate(base, TE_SIM, DEV, use_w, amp=USE_AMP, amp_dtype=AMP_DTYPE)
            pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res")},
                        open(f"{PERBUILDING_DIR}/{name}_{cond}.pkl","wb"))
            row={"model":name,"condition":cond,"weather":use_w,
                 "com_nrmse":round(res.get("Com NRMSE",float('nan')),3),"res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
                 "com_nmae":round(res.get("Com NMAE",float('nan')),3),"res_nmae":round(res.get("Res NMAE",float('nan')),3),
                 "com_crps":round(res.get("Com CRPS",float('nan')),3),"res_crps":round(res.get("Res CRPS",float('nan')),3),
                 "val_nll":(round(val,4) if val==val else ""),"params":params,"sec":round(time.time()-t0)}
            csv.DictWriter(open(SIM_CSV,"a",newline=""),FIELDS).writerow(row)
            print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}\n")
        except Exception as e:
            print(f"  FAILED {name}/{cond}: {type(e).__name__}: {e}"); traceback.print_exc()
print("simulated-eval DONE ->", SIM_CSV)

# ---- real-eval loop (no-weather only, same pattern as cell 7) ----
RFIELDS=["model","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","n_buildings","sec"]
rdone=set()
if os.path.getsize(REAL_CSV)>50:
    d=pd.read_csv(REAL_CSV); rdone={r.model for _,r in d.iterrows()}
PERBUILDING_REAL_DIR=f"{RESULTS_DIR}/perbuilding_real"; os.makedirs(PERBUILDING_REAL_DIR,exist_ok=True)

for name in list(V2_KW.keys()):
    if name in rdone: print(f"skip {name} (real-eval)"); continue
    t0=time.time()
    try:
        if name=="stackedensemble":
            base_nets=[]
            for bn in ENSEMBLE_BASES:
                if bn=="xgboost": continue
                bm=build(bn, L=L,H=H,n_time=TR["Ft"],n_weather=TR["Fw"],use_weather=False,**MODEL_KW[bn]).to(DEV)
                bm.load_state_dict(torch.load(f"{WEIGHTS_DIR}/{bn}_A.pt", map_location=DEV)); bm.eval()
                base_nets.append(bm)
            xgb_mdl,xgb_sig = pickle.load(open(f"{WEIGHTS_DIR}/xgboost_A.pkl","rb"))
            base=StackedEnsemble(base_nets, xgb_mdl, xgb_sig, L=L,H=H,**V2_KW["stackedensemble"]).to(DEV)
            base.load_state_dict(torch.load(f"{WEIGHTS_DIR}/stackedensemble_A.pt", map_location=DEV))
        else:
            base=build(name, L=L,H=H,n_time=TR["Ft"],n_weather=TR["Fw"],use_weather=False,**MODEL_KW[name]).to(DEV)
            base.load_state_dict(torch.load(f"{WEIGHTS_DIR}/{name}_A.pt", map_location=DEV))
        print(f"real-eval {name}...", flush=True)
        res=evaluate_real(base, TE_REAL, DEV, stride=24, max_windows_per_building=90, amp=USE_AMP, amp_dtype=AMP_DTYPE)
        pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res") if k in res},
                    open(f"{PERBUILDING_REAL_DIR}/{name}.pkl","wb"))
        row={"model":name,"com_nrmse":round(res.get("Com NRMSE",float('nan')),3),
             "res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
             "com_nmae":round(res.get("Com NMAE",float('nan')),3),"res_nmae":round(res.get("Res NMAE",float('nan')),3),
             "com_crps":round(res.get("Com CRPS",float('nan')),3),"res_crps":round(res.get("Res CRPS",float('nan')),3),
             "n_buildings":res.get("n_buildings",0),"sec":round(time.time()-t0)}
        csv.DictWriter(open(REAL_CSV,"a",newline=""),RFIELDS).writerow(row)
        print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}  (n={row['n_buildings']})\n")
    except Exception as e:
        print(f"  FAILED {name}: {type(e).__name__}: {e}"); traceback.print_exc()
print("real-eval DONE ->", REAL_CSV)

# ---- updated comparison table ----
sim = pd.read_csv(SIM_CSV); real = pd.read_csv(REAL_CSV)
print("\n=== SIMULATED (condition A, no-weather) ===")
for _,r in sim[sim.condition=="A"].sort_values("com_nrmse").iterrows():
    print(f"  {r.model:20s} Com {r.com_nrmse:7.3f} | Res {r.res_nrmse:7.3f}")
print("\n=== REAL (no-weather) ===")
for _,r in real.sort_values("com_nrmse").iterrows():
    print(f"  {r.model:20s} Com {r.com_nrmse:7.3f} | Res {r.res_nrmse:7.3f}")

models now: ['pwfnet', 'xpan', 'freqmoe', 'drdt', 'mscah', 'naive', 'linreg', 'lstm', 'gru', 'cnn_lstm', 'dlinear', 'persistence_ensemble', 'xgboost', 'timesnetxl', 'neuralgboosthybrid', 'robusttimesnet', 'stackedensemble']
train timesnetxl/A (7,038,448p)...
    ep000 val=0.9424 best=0.9424 bad=0
    ep010 val=0.4951 best=0.4951 bad=0
    ep020 val=0.4497 best=0.4497 bad=0
    ep030 val=0.4185 best=0.3890 bad=3
    ep040 val=0.4057 best=0.3759 bad=3
    ep050 val=0.3777 best=0.3665 bad=9
    ep060 val=0.3715 best=0.3301 bad=4
    ep070 val=0.3492 best=0.3301 bad=14
    ep080 val=0.3402 best=0.3281 bad=4
    ep090 val=0.3484 best=0.3101 bad=3
    ep100 val=0.3457 best=0.3011 bad=9
    ep110 val=0.3127 best=0.3011 bad=19
    early stop @ep111
  done 1246s  Com NRMSE=15.968  Res NRMSE=41.0

train timesnetxl/B (7,038,448p)...
    ep000 val=0.9424 best=0.9424 bad=0
    ep010 val=0.4951 best=0.4951 bad=0
    ep020 val=0.4497 best=0.4497 bad=0
    ep030 val=0.4185 best=0.3890 bad=3
    ep040 

KeyboardInterrupt: 

In [ ]:
# =====================================================================================
# APPEND: MegaLoadNet -- synthesizes TimesNet (periodic conv, proven strongest so far),
# iTransformer (variate-token attention), and TimeXer (single learnable global query
# cross-attends over branch outputs + variate tokens -- cheaper/more targeted than
# concatenation). Plus two load-forecasting-specific refinements validated by an
# independent 2025 PG&E/ESD load-forecasting competition study: per-hour output heads
# (not one shared head for all 24h) and a strictly-instantaneous degree-day weather
# response (elaborate temporal weather modeling didn't help in that study either).
# Mamba/selective-SSM deliberately excluded (too many nuances/bugs for this context).
# Assumes cells 1-8 (and 9-10 if run) have already executed in this session.
# =====================================================================================
import torch.nn as nn, torch.nn.functional as F

class MegaLoadNet(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=192,d_ff=384,heads=8,periods=(12,24,168),
                 temp_idx=None,dropout=0.15,dh=128,**kw):
        super().__init__()
        self.L,self.H,self.n_time,self.n_weather,self.use_weather=L,H,n_time,n_weather,use_weather
        self.seq=L+H; self.revin=RevIN(); self.temp_idx=temp_idx
        self.embed_a=nn.Linear(1,d_model)
        self.pos_a=nn.Parameter(torch.randn(1,self.seq,d_model)*0.02)
        self.blocks_a=nn.ModuleList([TimesBlock2(d_model,d_ff,p) for p in periods])
        self.norm_a=nn.LayerNorm(d_model); self.drop_a=nn.Dropout(dropout)
        self.pool_a=nn.Linear(self.seq*d_model, d_model)
        self.tok_load=nn.Linear(L,d_model)
        self.tok_cal=nn.ModuleList([nn.Linear(L,d_model) for _ in range(n_time)])
        self.tok_weather=nn.ModuleList([nn.Linear(L,d_model) for _ in range(n_weather)]) if use_weather else None
        self.tok_btype=nn.Linear(1,d_model)
        var_layer=nn.TransformerEncoderLayer(d_model,heads,d_ff,dropout,batch_first=True,activation='gelu')
        self.var_enc=nn.TransformerEncoder(var_layer,2)
        self.global_query=nn.Parameter(torch.randn(1,1,d_model)*0.02)
        self.fusion_attn=nn.MultiheadAttention(d_model,heads,dropout=dropout,batch_first=True)
        self.fusion_norm=nn.LayerNorm(d_model)
        if use_weather:
            if temp_idx is None: self.temp_proxy=nn.Linear(n_weather,1)
            self.Tbal_heat=nn.Parameter(torch.tensor(-0.3))
            self.Tbal_cool=nn.Parameter(torch.tensor(0.3))
            self.dd_mlp=nn.Sequential(nn.Linear(2,dh),nn.GELU(),nn.Linear(dh,1))
        self.hour_heads=nn.ModuleList([nn.Linear(d_model,2) for _ in range(H)])
    def _degree_day(self, exo):
        if not self.use_weather: return None
        weather=exo[...,self.n_time:self.n_time+self.n_weather]
        temp = weather[...,self.temp_idx] if self.temp_idx is not None else self.temp_proxy(weather).squeeze(-1)
        temp_fut=temp[:,self.L:]
        heat=F.relu(self.Tbal_heat-temp_fut); cool=F.relu(temp_fut-self.Tbal_cool)
        return self.dd_mlp(torch.stack([heat,cool],-1)).squeeze(-1)
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        full=torch.cat([yn,torch.zeros(B,self.H,device=yh.device,dtype=yn.dtype)],1)
        za=self.embed_a(full.unsqueeze(-1))+self.pos_a
        for blk in self.blocks_a: za=self.drop_a(self.norm_a(za+blk(za)))
        a_pooled=self.pool_a(za.reshape(B,-1)).unsqueeze(1)
        cal_hist=exo[:,:self.L,:self.n_time]
        toks=[self.tok_load(yn)]
        toks += [self.tok_cal[i](cal_hist[...,i]) for i in range(self.n_time)]
        if self.use_weather:
            w_hist=exo[:,:self.L,self.n_time:self.n_time+self.n_weather]
            toks += [self.tok_weather[i](w_hist[...,i]) for i in range(self.n_weather)]
        toks.append(self.tok_btype(exo[:,0,-1:]))
        var_tokens=torch.stack(toks,1)
        var_out=self.var_enc(var_tokens)
        b_load=var_out[:,0:1]
        kv=torch.cat([a_pooled, b_load, var_out], 1)
        q=self.global_query.expand(B,-1,-1)
        fused,_=self.fusion_attn(q, kv, kv)
        fused=self.fusion_norm(fused.squeeze(1))
        out=torch.stack([head(fused) for head in self.hour_heads],1)
        mu_n, raw = out[...,0], out[...,1]
        corr=self._degree_day(exo)
        if corr is not None: mu_n=mu_n+corr
        return mu_n, raw, mu, sd

MEGA_KW = {"megaloadnet": dict(d_model=192, d_ff=384, heads=8, periods=(12,24,168), temp_idx=0, dropout=0.15, dh=128)}
REG["megaloadnet"] = MegaLoadNet
MODEL_KW.update(MEGA_KW)
ALL_MODELS = ALL_MODELS + list(MEGA_KW.keys())
print(f"models now: {ALL_MODELS}")

import csv, time, pandas as pd, pickle, os, traceback
FIELDS=["model","condition","weather","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","val_nll","params","sec"]
done=set()
if os.path.getsize(SIM_CSV)>50:
    d=pd.read_csv(SIM_CSV); done={(r.model,r.condition) for _,r in d.iterrows()}
PERBUILDING_DIR=f"{RESULTS_DIR}/perbuilding_sim"; os.makedirs(PERBUILDING_DIR,exist_ok=True)

for name in list(MEGA_KW.keys()):
    for cond in CONDITIONS:
        if (name,cond) in done: print(f"skip {name}/{cond}"); continue
        use_w=CONDITIONS[cond]; t0=time.time()
        try:
            torch.manual_seed(SEED)
            base=build(name, L=L,H=H,n_time=TR["Ft"],n_weather=TR["Fw"],use_weather=use_w,**MODEL_KW[name]).to(DEV)
            params=count_params(base)
            print(f"train {name}/{cond} ({params:,}p)...", flush=True)
            val=train(base, base, TR, DEV, use_w, epochs=EPOCHS, patience=PATIENCE, steps=STEPS, bs=BS, lr=LR,
                      amp=USE_AMP, amp_dtype=AMP_DTYPE, clip=1.0, log=True)
            torch.save(base.state_dict(), f"{WEIGHTS_DIR}/{name}_{cond}.pt")   # save BEFORE eval
            res=evaluate(base, TE_SIM, DEV, use_w, amp=USE_AMP, amp_dtype=AMP_DTYPE)
            pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res")},
                        open(f"{PERBUILDING_DIR}/{name}_{cond}.pkl","wb"))
            row={"model":name,"condition":cond,"weather":use_w,
                 "com_nrmse":round(res.get("Com NRMSE",float('nan')),3),"res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
                 "com_nmae":round(res.get("Com NMAE",float('nan')),3),"res_nmae":round(res.get("Res NMAE",float('nan')),3),
                 "com_crps":round(res.get("Com CRPS",float('nan')),3),"res_crps":round(res.get("Res CRPS",float('nan')),3),
                 "val_nll":(round(val,4) if val==val else ""),"params":params,"sec":round(time.time()-t0)}
            csv.DictWriter(open(SIM_CSV,"a",newline=""),FIELDS).writerow(row)
            print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}\n")
        except Exception as e:
            print(f"  FAILED {name}/{cond}: {type(e).__name__}: {e}"); traceback.print_exc()
print("simulated-eval DONE ->", SIM_CSV)

RFIELDS=["model","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","n_buildings","sec"]
rdone=set()
if os.path.getsize(REAL_CSV)>50:
    d=pd.read_csv(REAL_CSV); rdone={r.model for _,r in d.iterrows()}
PERBUILDING_REAL_DIR=f"{RESULTS_DIR}/perbuilding_real"; os.makedirs(PERBUILDING_REAL_DIR,exist_ok=True)
for name in list(MEGA_KW.keys()):
    if name in rdone: print(f"skip {name} (real-eval)"); continue
    t0=time.time()
    try:
        base=build(name, L=L,H=H,n_time=TR["Ft"],n_weather=TR["Fw"],use_weather=False,**MODEL_KW[name]).to(DEV)
        base.load_state_dict(torch.load(f"{WEIGHTS_DIR}/{name}_A.pt", map_location=DEV))
        print(f"real-eval {name}...", flush=True)
        res=evaluate_real(base, TE_REAL, DEV, stride=24, max_windows_per_building=90, amp=USE_AMP, amp_dtype=AMP_DTYPE)
        pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res") if k in res},
                    open(f"{PERBUILDING_REAL_DIR}/{name}.pkl","wb"))
        row={"model":name,"com_nrmse":round(res.get("Com NRMSE",float('nan')),3),
             "res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
             "com_nmae":round(res.get("Com NMAE",float('nan')),3),"res_nmae":round(res.get("Res NMAE",float('nan')),3),
             "com_crps":round(res.get("Com CRPS",float('nan')),3),"res_crps":round(res.get("Res CRPS",float('nan')),3),
             "n_buildings":res.get("n_buildings",0),"sec":round(time.time()-t0)}
        csv.DictWriter(open(REAL_CSV,"a",newline=""),RFIELDS).writerow(row)
        print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}  (n={row['n_buildings']})\n")
    except Exception as e:
        print(f"  FAILED {name}: {type(e).__name__}: {e}"); traceback.print_exc()
print("real-eval DONE ->", REAL_CSV)

sim = pd.read_csv(SIM_CSV); real = pd.read_csv(REAL_CSV)
print("\n=== SIMULATED (condition A) ==="); [print(f"  {r.model:16s} Com {r.com_nrmse:7.3f} | Res {r.res_nrmse:7.3f}") for _,r in sim[sim.condition=="A"].sort_values("com_nrmse").iterrows()]
print("\n=== REAL (no-weather) ==="); [print(f"  {r.model:16s} Com {r.com_nrmse:7.3f} | Res {r.res_nrmse:7.3f}") for _,r in real.sort_values("com_nrmse").iterrows()]

models now: ['pwfnet', 'xpan', 'freqmoe', 'drdt', 'mscah', 'naive', 'linreg', 'lstm', 'gru', 'cnn_lstm', 'dlinear', 'persistence_ensemble', 'xgboost', 'megaloadnet']
train megaloadnet/A (12,078,384p)...
    ep000 val=0.7947 best=0.7947 bad=0
    ep010 val=0.5185 best=0.4882 bad=3
    ep020 val=0.4651 best=0.4579 bad=9
    ep030 val=0.4113 best=0.4113 bad=0
    ep040 val=0.4373 best=0.4030 bad=2
    ep050 val=0.4067 best=0.3752 bad=1
    ep060 val=0.3807 best=0.3672 bad=2
    ep070 val=0.3332 best=0.3332 bad=0
    ep080 val=0.3596 best=0.3332 bad=10
    ep090 val=0.3482 best=0.3332 bad=20
    early stop @ep90
  done 764s  Com NRMSE=15.797  Res NRMSE=40.037

train megaloadnet/B (12,306,035p)...
    ep000 val=0.7662 best=0.7662 bad=0
    ep010 val=0.4973 best=0.4303 bad=1
    ep020 val=0.4142 best=0.3967 bad=2
    ep030 val=0.3855 best=0.3692 bad=1
    ep040 val=0.3469 best=0.3388 bad=1
    ep050 val=0.3740 best=0.3266 bad=3
    ep060 val=0.3226 best=0.2956 bad=5
    ep070 val=0.2987 best

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [ ]:
# =====================================================================================
# APPEND: 2 transformer-upgrade + 2 TimesNet + 2 combination models.
#   timexerload       -- faithful TimeXer: patch endo self-attn + ONE learnable global
#                        token cross-attending to per-variate exogenous tokens. ~5M/5.4M.
#   crossformerload   -- faithful Crossformer: DSW segment embedding + stacked Two-Stage
#                        Attention (cross-time self-attn per variate, then router-based
#                        cross-dimension gather/distribute, O(D) not O(D^2)). ~11.4M/15.6M.
#   timesnetxl_full   -- real TimesNet fully: multi-kernel Inception block AND FFT-
#                        amplitude-weighted period aggregation together. ~17.7M.
#   timesnetdeep      -- genuinely stacked LAYERS of period-blocks (paper's actual deep
#                        residual stacking; earlier variants only had one layer). ~11.7M.
#   timexerperiodic   -- combination: TimesNet's periodic-conv GENERATES the endogenous
#                        tokens, feeding TimeXer's global-token exo-bridging mechanism.
#   crossperiodic     -- combination: TimesNet's periodic-conv builds the load token,
#                        then Crossformer's router-based cross-dimension stage fuses it
#                        with the other variates.
# Assumes cells 1-8 (and 9-10 if run) have already executed in this session.
# =====================================================================================
import torch.nn as nn, torch.nn.functional as F

class InceptionBlock2D(nn.Module):
    def __init__(self, d_model, d_ff, period, kernels=(1,3,5,7)):
        super().__init__(); self.period = period
        self.branch1 = nn.ModuleList([nn.Conv2d(d_model,d_ff,k,padding=k//2) for k in kernels])
        self.act = nn.GELU()
        self.branch2 = nn.ModuleList([nn.Conv2d(d_ff,d_model,k,padding=k//2) for k in kernels])
    def forward(self, x):
        B,T,D = x.shape; p=self.period
        pad=(p-T%p)%p; xp=F.pad(x,(0,0,0,pad)) if pad else x
        n=xp.size(1)//p; g=xp.reshape(B,n,p,D).permute(0,3,1,2)
        h = self.act(sum(b(g) for b in self.branch1))
        out = sum(b(h) for b in self.branch2)
        return out.permute(0,2,3,1).reshape(B,n*p,D)[:,:T]

def fft_amp_weights(x, periods):
    xf = torch.fft.rfft(x.float(), dim=1).abs().mean(dim=(0,2))
    T = x.size(1)
    idx = torch.tensor([max(1, round(T/p)) for p in periods], device=x.device).clamp(max=xf.size(0)-1)
    return F.softmax(xf[idx], dim=0)

class TwoStageAttentionLayer(nn.Module):
    def __init__(self, d_model, heads, n_seg, D, n_routers=4, d_ff=None, dropout=0.1):
        super().__init__(); d_ff = d_ff or d_model*2
        self.time_attn = nn.MultiheadAttention(d_model, heads, dropout=dropout, batch_first=True)
        self.time_norm1 = nn.LayerNorm(d_model); self.time_ff = nn.Sequential(
            nn.Linear(d_model,d_ff), nn.GELU(), nn.Linear(d_ff,d_model)); self.time_norm2 = nn.LayerNorm(d_model)
        self.routers = nn.Parameter(torch.randn(1, n_seg, n_routers, d_model)*0.02)
        self.gather_attn = nn.MultiheadAttention(d_model, heads, dropout=dropout, batch_first=True)
        self.distrib_attn = nn.MultiheadAttention(d_model, heads, dropout=dropout, batch_first=True)
        self.dim_norm1 = nn.LayerNorm(d_model); self.dim_ff = nn.Sequential(
            nn.Linear(d_model,d_ff), nn.GELU(), nn.Linear(d_ff,d_model)); self.dim_norm2 = nn.LayerNorm(d_model)
    def forward(self, x):
        B,D,n_seg,d = x.shape
        xt = x.reshape(B*D, n_seg, d)
        a,_ = self.time_attn(xt,xt,xt); xt = self.time_norm1(xt+a); xt = self.time_norm2(xt + self.time_ff(xt))
        x = xt.reshape(B,D,n_seg,d)
        xd = x.permute(0,2,1,3).reshape(B*n_seg, D, d)
        r = self.routers.expand(B,-1,-1,-1).reshape(B*n_seg, -1, d)
        gathered,_ = self.gather_attn(r, xd, xd)
        distributed,_ = self.distrib_attn(xd, gathered, gathered)
        xd = self.dim_norm1(xd + distributed); xd = self.dim_norm2(xd + self.dim_ff(xd))
        return xd.reshape(B,n_seg,D,d).permute(0,2,1,3)

class _CrossDimOnly(nn.Module):
    def __init__(self, d_model, heads, n_routers, dropout):
        super().__init__()
        self.gather_attn=nn.MultiheadAttention(d_model,heads,dropout=dropout,batch_first=True)
        self.distrib_attn=nn.MultiheadAttention(d_model,heads,dropout=dropout,batch_first=True)
        self.norm1=nn.LayerNorm(d_model); self.norm2=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(nn.Linear(d_model,d_model*2),nn.GELU(),nn.Linear(d_model*2,d_model))
        self.routers=nn.Parameter(torch.randn(1,n_routers,d_model)*0.02)
    def forward(self, x):
        B,D,d=x.shape; r=self.routers.expand(B,-1,-1)
        gathered,_=self.gather_attn(r,x,x); distributed,_=self.distrib_attn(x,gathered,gathered)
        x=self.norm1(x+distributed); x=self.norm2(x+self.ff(x))
        return x

class TimeXerLoad(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=320,heads=10,layers=5,d_ff=640,patch=24,stride=12,dropout=0.15,**kw):
        super().__init__()
        self.L,self.H=L,H; self.n_time,self.n_weather,self.use_weather=n_time,n_weather,use_weather
        self.revin=RevIN(); self.patch,self.stride=patch,stride; self.np=(L-patch)//stride+1
        self.patch_embed=nn.Linear(patch,d_model)
        self.pos=nn.Parameter(torch.randn(1,self.np,d_model)*0.02)
        self.global_tok=nn.Parameter(torch.randn(1,1,d_model)*0.02)
        enc=nn.TransformerEncoderLayer(d_model,heads,d_ff,dropout,batch_first=True,activation='gelu')
        self.endo_enc=nn.TransformerEncoder(enc,layers)
        self.tok_cal=nn.ModuleList([nn.Linear(L,d_model) for _ in range(n_time)])
        self.tok_weather=nn.ModuleList([nn.Linear(L,d_model) for _ in range(n_weather)]) if use_weather else None
        self.tok_btype=nn.Linear(1,d_model)
        self.exo_cross=nn.MultiheadAttention(d_model,heads,dropout=dropout,batch_first=True)
        self.exo_norm=nn.LayerNorm(d_model)
        self.fut_proj=nn.Sequential(nn.Linear(H*(n_time+(n_weather if use_weather else 0)+1),d_model),nn.GELU())
        self.head=nn.Sequential(nn.LayerNorm(2*d_model),nn.Linear(2*d_model,2*H))
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        patches=self.patch_embed(yn.unfold(1,self.patch,self.stride))+self.pos
        seq=torch.cat([self.global_tok.expand(B,-1,-1), patches],1)
        seq=self.endo_enc(seq); g=seq[:,0:1]
        cal_hist=exo[:,:self.L,:self.n_time]
        toks=[self.tok_cal[i](cal_hist[...,i]) for i in range(self.n_time)]
        if self.use_weather:
            w_hist=exo[:,:self.L,self.n_time:self.n_time+self.n_weather]
            toks+=[self.tok_weather[i](w_hist[...,i]) for i in range(self.n_weather)]
        toks.append(self.tok_btype(exo[:,0,-1:]))
        exo_tok=torch.stack(toks,1)
        bridged,_=self.exo_cross(g, exo_tok, exo_tok)
        fused=self.exo_norm(g+bridged).squeeze(1)
        fut=self.fut_proj(exo[:,self.L:].reshape(B,-1))
        out=self.head(torch.cat([fused,fut],-1)).view(B,self.H,2)
        return out[...,0],out[...,1],mu,sd

class CrossformerLoad(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=288,heads=8,layers=4,seg_len=24,n_routers=4,dropout=0.15,**kw):
        super().__init__()
        self.L,self.H=L,H; self.n_time,self.n_weather,self.use_weather=n_time,n_weather,use_weather
        self.revin=RevIN(); self.seg=seg_len; self.n_seg=L//seg_len
        self.D=1+n_time+(n_weather if use_weather else 0)+1
        self.seg_embed=nn.Linear(seg_len,d_model)
        self.pos=nn.Parameter(torch.randn(1,self.D,self.n_seg,d_model)*0.02)
        self.tsa=nn.ModuleList([TwoStageAttentionLayer(d_model,heads,self.n_seg,self.D,n_routers,dropout=dropout) for _ in range(layers)])
        self.pool=nn.Linear(self.D*self.n_seg*d_model, d_model)
        self.fut_proj=nn.Sequential(nn.Linear(H*(n_time+(n_weather if use_weather else 0)+1),d_model),nn.GELU())
        self.head=nn.Sequential(nn.LayerNorm(2*d_model),nn.Linear(2*d_model,2*H))
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        cal=exo[:,:self.L,:self.n_time].transpose(1,2)
        parts=[yn.unsqueeze(1), cal]
        if self.use_weather:
            w=exo[:,:self.L,self.n_time:self.n_time+self.n_weather].transpose(1,2); parts.append(w)
        bt=exo[:,:self.L,-1:].transpose(1,2); parts.append(bt)
        allvar=torch.cat(parts,1)
        segs=allvar.reshape(B,self.D,self.n_seg,self.seg)
        z=self.seg_embed(segs)+self.pos
        for layer in self.tsa: z=layer(z)
        pooled=self.pool(z.reshape(B,-1))
        fut=self.fut_proj(exo[:,self.L:].reshape(B,-1))
        out=self.head(torch.cat([pooled,fut],-1)).view(B,self.H,2)
        return out[...,0],out[...,1],mu,sd

class TimesNetXL_Full(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=128,d_ff=256,periods=(12,24,168),dropout=0.15,**kw):
        super().__init__()
        self.L,self.H=L,H; self.seq=L+H; self.revin=RevIN(); self.periods=periods
        self.embed=nn.Linear(1,d_model); self.pos=nn.Parameter(torch.randn(1,self.seq,d_model)*0.02)
        self.blocks=nn.ModuleList([InceptionBlock2D(d_model,d_ff,p) for p in periods])
        self.norm=nn.LayerNorm(d_model); self.drop=nn.Dropout(dropout)
        self.head=nn.Linear(self.seq*d_model, 2*H)
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        full=torch.cat([yn,torch.zeros(B,self.H,device=yh.device,dtype=yn.dtype)],1)
        z=self.embed(full.unsqueeze(-1))+self.pos
        w=fft_amp_weights(yn.unsqueeze(-1), self.periods)
        for wi, blk in zip(w, self.blocks): z=self.drop(self.norm(z+wi*blk(z)))
        out=self.head(z.reshape(B,-1)).view(B,self.H,2)
        return out[...,0],out[...,1],mu,sd

class TimesNetDeep(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=176,d_ff=352,periods=(12,24,168),n_layers=3,dropout=0.15,**kw):
        super().__init__()
        self.L,self.H=L,H; self.seq=L+H; self.revin=RevIN()
        self.embed=nn.Linear(1,d_model); self.pos=nn.Parameter(torch.randn(1,self.seq,d_model)*0.02)
        self.layers=nn.ModuleList([nn.ModuleList([TimesBlock2(d_model,d_ff,p) for p in periods]) for _ in range(n_layers)])
        self.norms=nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.drop=nn.Dropout(dropout)
        self.head=nn.Linear(self.seq*d_model, 2*H)
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        full=torch.cat([yn,torch.zeros(B,self.H,device=yh.device,dtype=yn.dtype)],1)
        z=self.embed(full.unsqueeze(-1))+self.pos
        for layer_blocks, norm in zip(self.layers, self.norms):
            layer_out = sum(blk(z) for blk in layer_blocks) / len(layer_blocks)
            z = self.drop(norm(z + layer_out))
        out=self.head(z.reshape(B,-1)).view(B,self.H,2)
        return out[...,0],out[...,1],mu,sd

class TimeXerPeriodic(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=192,heads=8,d_ff=384,periods=(12,24,168),dropout=0.15,**kw):
        super().__init__()
        self.L,self.H=L,H; self.n_time,self.n_weather,self.use_weather=n_time,n_weather,use_weather
        self.revin=RevIN()
        self.embed=nn.Linear(1,d_model); self.pos=nn.Parameter(torch.randn(1,L,d_model)*0.02)
        self.blocks=nn.ModuleList([TimesBlock2(d_model,d_ff,p) for p in periods])
        self.norm=nn.LayerNorm(d_model); self.drop=nn.Dropout(dropout)
        self.global_tok=nn.Parameter(torch.randn(1,1,d_model)*0.02)
        self.endo_attn=nn.MultiheadAttention(d_model,heads,dropout=dropout,batch_first=True)
        self.tok_cal=nn.ModuleList([nn.Linear(L,d_model) for _ in range(n_time)])
        self.tok_weather=nn.ModuleList([nn.Linear(L,d_model) for _ in range(n_weather)]) if use_weather else None
        self.tok_btype=nn.Linear(1,d_model)
        self.exo_cross=nn.MultiheadAttention(d_model,heads,dropout=dropout,batch_first=True)
        self.exo_norm=nn.LayerNorm(d_model)
        self.fut_proj=nn.Sequential(nn.Linear(H*(n_time+(n_weather if use_weather else 0)+1),d_model),nn.GELU())
        self.head=nn.Sequential(nn.LayerNorm(2*d_model),nn.Linear(2*d_model,2*H))
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        z=self.embed(yn.unsqueeze(-1))+self.pos
        for blk in self.blocks: z=self.drop(self.norm(z+blk(z)))
        g0=self.global_tok.expand(B,-1,-1)
        g,_=self.endo_attn(g0, z, z)
        cal_hist=exo[:,:self.L,:self.n_time]
        toks=[self.tok_cal[i](cal_hist[...,i]) for i in range(self.n_time)]
        if self.use_weather:
            w_hist=exo[:,:self.L,self.n_time:self.n_time+self.n_weather]
            toks+=[self.tok_weather[i](w_hist[...,i]) for i in range(self.n_weather)]
        toks.append(self.tok_btype(exo[:,0,-1:]))
        exo_tok=torch.stack(toks,1)
        bridged,_=self.exo_cross(g, exo_tok, exo_tok)
        fused=self.exo_norm(g+bridged).squeeze(1)
        fut=self.fut_proj(exo[:,self.L:].reshape(B,-1))
        out=self.head(torch.cat([fused,fut],-1)).view(B,self.H,2)
        return out[...,0],out[...,1],mu,sd

class CrossPeriodic(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=192,heads=8,d_ff=384,periods=(12,24,168),n_routers=4,dropout=0.15,**kw):
        super().__init__()
        self.L,self.H=L,H; self.n_time,self.n_weather,self.use_weather=n_time,n_weather,use_weather
        self.revin=RevIN()
        self.load_embed=nn.Linear(1,d_model); self.load_pos=nn.Parameter(torch.randn(1,L,d_model)*0.02)
        self.load_blocks=nn.ModuleList([TimesBlock2(d_model,d_ff,p) for p in periods])
        self.load_norm=nn.LayerNorm(d_model); self.drop=nn.Dropout(dropout)
        self.load_pool=nn.Linear(L*d_model, d_model)
        self.tok_cal=nn.ModuleList([nn.Linear(L,d_model) for _ in range(n_time)])
        self.tok_weather=nn.ModuleList([nn.Linear(L,d_model) for _ in range(n_weather)]) if use_weather else None
        self.tok_btype=nn.Linear(1,d_model)
        self.cross_dim=_CrossDimOnly(d_model, heads, n_routers, dropout)
        self.fut_proj=nn.Sequential(nn.Linear(H*(n_time+(n_weather if use_weather else 0)+1),d_model),nn.GELU())
        self.head=nn.Sequential(nn.LayerNorm(2*d_model),nn.Linear(2*d_model,2*H))
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        zl=self.load_embed(yn.unsqueeze(-1))+self.load_pos
        for blk in self.load_blocks: zl=self.drop(self.load_norm(zl+blk(zl)))
        load_tok=self.load_pool(zl.reshape(B,-1)).unsqueeze(1)
        cal_hist=exo[:,:self.L,:self.n_time]
        toks=[load_tok.squeeze(1)]+[self.tok_cal[i](cal_hist[...,i]) for i in range(self.n_time)]
        if self.use_weather:
            w_hist=exo[:,:self.L,self.n_time:self.n_time+self.n_weather]
            toks+=[self.tok_weather[i](w_hist[...,i]) for i in range(self.n_weather)]
        toks.append(self.tok_btype(exo[:,0,-1:]))
        var_tokens=torch.stack(toks,1)
        fused_all=self.cross_dim(var_tokens)
        fused=fused_all[:,0]
        fut=self.fut_proj(exo[:,self.L:].reshape(B,-1))
        out=self.head(torch.cat([fused,fut],-1)).view(B,self.H,2)
        return out[...,0],out[...,1],mu,sd

V4_KW = {
  "timexerload":     dict(d_model=320, heads=10, layers=5, d_ff=640, patch=24, stride=12, dropout=0.15),
  "crossformerload": dict(d_model=288, heads=8, layers=4, seg_len=24, n_routers=4, dropout=0.15),
  "timesnetxl_full":  dict(d_model=128, d_ff=256, periods=(12,24,168), dropout=0.15),
  "timesnetdeep":     dict(d_model=176, d_ff=352, periods=(12,24,168), n_layers=3, dropout=0.15),
  "timexerperiodic":  dict(d_model=192, heads=8, d_ff=384, periods=(12,24,168), dropout=0.15),
  "crossperiodic":    dict(d_model=192, heads=8, d_ff=384, periods=(12,24,168), n_routers=4, dropout=0.15),
}
V4_REG = {"timexerload":TimeXerLoad, "crossformerload":CrossformerLoad, "timesnetxl_full":TimesNetXL_Full,
          "timesnetdeep":TimesNetDeep, "timexerperiodic":TimeXerPeriodic, "crossperiodic":CrossPeriodic}
REG.update(V4_REG)
MODEL_KW.update(V4_KW)
ALL_MODELS = ALL_MODELS + list(V4_KW.keys())
print(f"models now: {ALL_MODELS}")

import csv, time, pandas as pd, pickle, os, traceback
FIELDS=["model","condition","weather","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","val_nll","params","sec"]
done=set()
if os.path.getsize(SIM_CSV)>50:
    d=pd.read_csv(SIM_CSV); done={(r.model,r.condition) for _,r in d.iterrows()}
PERBUILDING_DIR=f"{RESULTS_DIR}/perbuilding_sim"; os.makedirs(PERBUILDING_DIR,exist_ok=True)

for name in list(V4_KW.keys()):
    for cond in CONDITIONS:
        if (name,cond) in done: print(f"skip {name}/{cond}"); continue
        use_w=CONDITIONS[cond]; t0=time.time()
        try:
            torch.manual_seed(SEED)
            base=build(name, L=L,H=H,n_time=TR["Ft"],n_weather=TR["Fw"],use_weather=use_w,**MODEL_KW[name]).to(DEV)
            params=count_params(base)
            print(f"train {name}/{cond} ({params:,}p)...", flush=True)
            val=train(base, base, TR, DEV, use_w, epochs=EPOCHS, patience=PATIENCE, steps=STEPS, bs=BS, lr=LR,
                      amp=USE_AMP, amp_dtype=AMP_DTYPE, clip=1.0, log=True)
            torch.save(base.state_dict(), f"{WEIGHTS_DIR}/{name}_{cond}.pt")
            res=evaluate(base, TE_SIM, DEV, use_w, amp=USE_AMP, amp_dtype=AMP_DTYPE)
            pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res")},
                        open(f"{PERBUILDING_DIR}/{name}_{cond}.pkl","wb"))
            row={"model":name,"condition":cond,"weather":use_w,
                 "com_nrmse":round(res.get("Com NRMSE",float('nan')),3),"res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
                 "com_nmae":round(res.get("Com NMAE",float('nan')),3),"res_nmae":round(res.get("Res NMAE",float('nan')),3),
                 "com_crps":round(res.get("Com CRPS",float('nan')),3),"res_crps":round(res.get("Res CRPS",float('nan')),3),
                 "val_nll":(round(val,4) if val==val else ""),"params":params,"sec":round(time.time()-t0)}
            csv.DictWriter(open(SIM_CSV,"a",newline=""),FIELDS).writerow(row)
            print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}\n")
        except Exception as e:
            print(f"  FAILED {name}/{cond}: {type(e).__name__}: {e}"); traceback.print_exc()
print("simulated-eval DONE ->", SIM_CSV)

RFIELDS=["model","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","n_buildings","sec"]
rdone=set()
if os.path.getsize(REAL_CSV)>50:
    d=pd.read_csv(REAL_CSV); rdone={r.model for _,r in d.iterrows()}
PERBUILDING_REAL_DIR=f"{RESULTS_DIR}/perbuilding_real"; os.makedirs(PERBUILDING_REAL_DIR,exist_ok=True)
for name in list(V4_KW.keys()):
    if name in rdone: print(f"skip {name} (real-eval)"); continue
    t0=time.time()
    try:
        base=build(name, L=L,H=H,n_time=TR["Ft"],n_weather=TR["Fw"],use_weather=False,**MODEL_KW[name]).to(DEV)
        base.load_state_dict(torch.load(f"{WEIGHTS_DIR}/{name}_A.pt", map_location=DEV))
        print(f"real-eval {name}...", flush=True)
        res=evaluate_real(base, TE_REAL, DEV, stride=24, max_windows_per_building=90, amp=USE_AMP, amp_dtype=AMP_DTYPE)
        pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res") if k in res},
                    open(f"{PERBUILDING_REAL_DIR}/{name}.pkl","wb"))
        row={"model":name,"com_nrmse":round(res.get("Com NRMSE",float('nan')),3),
             "res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
             "com_nmae":round(res.get("Com NMAE",float('nan')),3),"res_nmae":round(res.get("Res NMAE",float('nan')),3),
             "com_crps":round(res.get("Com CRPS",float('nan')),3),"res_crps":round(res.get("Res CRPS",float('nan')),3),
             "n_buildings":res.get("n_buildings",0),"sec":round(time.time()-t0)}
        csv.DictWriter(open(REAL_CSV,"a",newline=""),RFIELDS).writerow(row)
        print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}  (n={row['n_buildings']})\n")
    except Exception as e:
        print(f"  FAILED {name}: {type(e).__name__}: {e}"); traceback.print_exc()
print("real-eval DONE ->", REAL_CSV)

sim = pd.read_csv(SIM_CSV); real = pd.read_csv(REAL_CSV)
print("\n=== SIMULATED (condition A) ==="); [print(f"  {r.model:18s} Com {r.com_nrmse:7.3f} | Res {r.res_nrmse:7.3f}") for _,r in sim[sim.condition=="A"].sort_values("com_nrmse").iterrows()]
print("\n=== REAL (no-weather) ==="); [print(f"  {r.model:18s} Com {r.com_nrmse:7.3f} | Res {r.res_nrmse:7.3f}") for _,r in real.sort_values("com_nrmse").iterrows()]








models now: ['pwfnet', 'xpan', 'freqmoe', 'drdt', 'mscah', 'naive', 'linreg', 'lstm', 'gru', 'cnn_lstm', 'dlinear', 'persistence_ensemble', 'xgboost', 'megaloadnet', 'timexerload', 'crossformerload', 'timesnetxl_full', 'timesnetdeep', 'timexerperiodic', 'crossperiodic']
train timexerload/A (4,948,848p)...
    ep000 val=0.8093 best=0.8093 bad=0
    ep010 val=0.5678 best=0.5678 bad=0
    ep020 val=0.5285 best=0.5208 bad=6
    ep030 val=0.5315 best=0.4952 bad=7
    ep040 val=0.5556 best=0.4809 bad=7
    ep050 val=0.5597 best=0.4596 bad=7
    ep060 val=0.4685 best=0.4596 bad=17
    early stop @ep63
  done 403s  Com NRMSE=17.829  Res NRMSE=43.107

train timexerload/B (5,381,168p)...
    ep000 val=0.8338 best=0.8338 bad=0
    ep010 val=0.5303 best=0.5210 bad=1
    ep020 val=0.4328 best=0.4328 bad=0
    ep030 val=0.4302 best=0.4128 bad=5
    ep040 val=0.4613 best=0.4009 bad=7
    ep050 val=0.3959 best=0.3407 bad=4
    ep060 val=0.3602 best=0.3407 bad=14
    ep070 val=0.3746 best=0.3225 bad=5


KeyboardInterrupt: 

In [ ]:
# =====================================================================================
# APPEND: vanillatransformerload -- the plain transformer, no patching. Each hour is its
# own token (L=168 tokens), standard learned positional encoding, standard multi-head
# self-attention. Exogenous features enter only via a simple flattened future-projection.
# =====================================================================================
import torch.nn as nn

class VanillaTransformerLoad(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=256,heads=8,layers=4,d_ff=512,dropout=0.15,**kw):
        super().__init__()
        self.L,self.H=L,H
        c_exo = n_time+(n_weather if use_weather else 0)+1
        self.revin=RevIN()
        self.embed=nn.Linear(1,d_model)
        self.pos=nn.Parameter(torch.randn(1,L,d_model)*0.02)
        enc=nn.TransformerEncoderLayer(d_model,heads,d_ff,dropout,batch_first=True,activation='gelu')
        self.encoder=nn.TransformerEncoder(enc,layers)
        self.exo_fut=nn.Sequential(nn.Linear(H*c_exo,d_model),nn.GELU())
        self.head=nn.Sequential(nn.LayerNorm(L*d_model+d_model),nn.Linear(L*d_model+d_model,2*H))
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        z=self.embed(yn.unsqueeze(-1))+self.pos
        z=self.encoder(z).reshape(B,-1)
        z=torch.cat([z,self.exo_fut(exo[:,self.L:].reshape(B,-1))],-1)
        o=self.head(z).view(B,self.H,2); return o[...,0],o[...,1],mu,sd

VAN_KW = {"vanillatransformerload": dict(d_model=256, heads=8, layers=4, d_ff=512, dropout=0.15)}
REG["vanillatransformerload"] = VanillaTransformerLoad
MODEL_KW.update(VAN_KW)
ALL_MODELS = ALL_MODELS + list(VAN_KW.keys())
print(f"models now: {ALL_MODELS}")

import csv, time, pandas as pd, pickle, os, traceback
FIELDS=["model","condition","weather","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","val_nll","params","sec"]
done=set()
if os.path.getsize(SIM_CSV)>50:
    d=pd.read_csv(SIM_CSV); done={(r.model,r.condition) for _,r in d.iterrows()}
PERBUILDING_DIR=f"{RESULTS_DIR}/perbuilding_sim"; os.makedirs(PERBUILDING_DIR,exist_ok=True)

for name in list(VAN_KW.keys()):
    for cond in CONDITIONS:
        if (name,cond) in done: print(f"skip {name}/{cond}"); continue
        use_w=CONDITIONS[cond]; t0=time.time()
        try:
            torch.manual_seed(SEED)
            base=build(name, L=L,H=H,n_time=TR["Ft"],n_weather=TR["Fw"],use_weather=use_w,**MODEL_KW[name]).to(DEV)
            params=count_params(base)
            print(f"train {name}/{cond} ({params:,}p)...", flush=True)
            val=train(base, base, TR, DEV, use_w, epochs=EPOCHS, patience=PATIENCE, steps=STEPS, bs=BS, lr=LR,
                      amp=USE_AMP, amp_dtype=AMP_DTYPE, clip=1.0, log=True)
            torch.save(base.state_dict(), f"{WEIGHTS_DIR}/{name}_{cond}.pt")
            res=evaluate(base, TE_SIM, DEV, use_w, amp=USE_AMP, amp_dtype=AMP_DTYPE)
            pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res")},
                        open(f"{PERBUILDING_DIR}/{name}_{cond}.pkl","wb"))
            row={"model":name,"condition":cond,"weather":use_w,
                 "com_nrmse":round(res.get("Com NRMSE",float('nan')),3),"res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
                 "com_nmae":round(res.get("Com NMAE",float('nan')),3),"res_nmae":round(res.get("Res NMAE",float('nan')),3),
                 "com_crps":round(res.get("Com CRPS",float('nan')),3),"res_crps":round(res.get("Res CRPS",float('nan')),3),
                 "val_nll":(round(val,4) if val==val else ""),"params":params,"sec":round(time.time()-t0)}
            csv.DictWriter(open(SIM_CSV,"a",newline=""),FIELDS).writerow(row)
            print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}\n")
        except Exception as e:
            print(f"  FAILED {name}/{cond}: {type(e).__name__}: {e}"); traceback.print_exc()
print("simulated-eval DONE ->", SIM_CSV)
# =====================================================================================
# APPEND: 3 design iterations on crossformerload_final.
#   crossformerload_multiscale  -- runs DSW+TSA at two segment lengths (12h, 24h) in
#                                   parallel, fused (Crossformer + TimesNet's multi-scale idea)
#   crossformerload_dlinearskip -- adds a DLinear-style trend+seasonal ADDITIVE residual
#                                   alongside the Crossformer output (predict + correct)
#   crossformerload_varseg      -- load variate gets seg_len=12 (finer), all other variates
#                                   get seg_len=24 (coarser), two separate pipelines fused
# All 3 also include the degree-day branch and per-hour heads from crossformerload_final.
# =====================================================================================
import torch.nn as nn, torch.nn.functional as F

class CrossformerLoad_Multiscale(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=224,heads=8,layers=3,seg_lens=(12,24),n_routers=4,dropout=0.15,
                 temp_idx=None,dh=128,**kw):
        super().__init__()
        self.L,self.H=L,H; self.n_time,self.n_weather,self.use_weather=n_time,n_weather,use_weather
        self.revin=RevIN(); self.seg_lens=seg_lens
        self.D=1+n_time+(n_weather if use_weather else 0)+1
        self.n_segs=[L//s for s in seg_lens]
        self.seg_embeds=nn.ModuleList([nn.Linear(s,d_model) for s in seg_lens])
        self.poss=nn.ParameterList([nn.Parameter(torch.randn(1,self.D,ns,d_model)*0.02) for ns in self.n_segs])
        self.tsas=nn.ModuleList([nn.ModuleList([TwoStageAttentionLayer(d_model,heads,ns,self.D,n_routers,dropout=dropout) for _ in range(layers)]) for ns in self.n_segs])
        self.pools=nn.ModuleList([nn.Linear(self.D*ns*d_model, d_model) for ns in self.n_segs])
        self.fut_proj=nn.Sequential(nn.Linear(H*(n_time+(n_weather if use_weather else 0)+1),d_model),nn.GELU())
        self.fuse_norm=nn.LayerNorm(len(seg_lens)*d_model+d_model)
        self.hour_heads=nn.ModuleList([nn.Linear(len(seg_lens)*d_model+d_model,2) for _ in range(H)])
        self.temp_idx=temp_idx
        if use_weather:
            if temp_idx is None: self.temp_proxy=nn.Linear(n_weather,1)
            self.Tbal_heat=nn.Parameter(torch.tensor(-0.3)); self.Tbal_cool=nn.Parameter(torch.tensor(0.3))
            self.dd_mlp=nn.Sequential(nn.Linear(2,dh),nn.GELU(),nn.Linear(dh,1))
    def _degree_day(self, exo):
        if not self.use_weather: return None
        weather=exo[...,self.n_time:self.n_time+self.n_weather]
        temp = weather[...,self.temp_idx] if self.temp_idx is not None else self.temp_proxy(weather).squeeze(-1)
        temp_fut=temp[:,self.L:]
        heat=F.relu(self.Tbal_heat-temp_fut); cool=F.relu(temp_fut-self.Tbal_cool)
        return self.dd_mlp(torch.stack([heat,cool],-1)).squeeze(-1)
    def _allvar(self, yn, exo):
        cal=exo[:,:self.L,:self.n_time].transpose(1,2)
        parts=[yn.unsqueeze(1), cal]
        if self.use_weather:
            w=exo[:,:self.L,self.n_time:self.n_time+self.n_weather].transpose(1,2); parts.append(w)
        bt=exo[:,:self.L,-1:].transpose(1,2); parts.append(bt)
        return torch.cat(parts,1)
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        allvar=self._allvar(yn,exo)
        pooled_list=[]
        for i,(seg,ns) in enumerate(zip(self.seg_lens,self.n_segs)):
            segs=allvar.reshape(B,self.D,ns,seg)
            z=self.seg_embeds[i](segs)+self.poss[i]
            for layer in self.tsas[i]: z=layer(z)
            pooled_list.append(self.pools[i](z.reshape(B,-1)))
        fut=self.fut_proj(exo[:,self.L:].reshape(B,-1))
        fused=self.fuse_norm(torch.cat(pooled_list+[fut],-1))
        out=torch.stack([head(fused) for head in self.hour_heads],1)
        mu_n, raw = out[...,0], out[...,1]
        corr=self._degree_day(exo)
        if corr is not None: mu_n=mu_n+corr
        return mu_n, raw, mu, sd

class CrossformerLoad_DLinearSkip(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=288,heads=8,layers=4,seg_len=24,n_routers=4,dropout=0.15,
                 temp_idx=None,dh=128,kernel=25,**kw):
        super().__init__()
        self.L,self.H=L,H; self.n_time,self.n_weather,self.use_weather=n_time,n_weather,use_weather
        self.revin=RevIN(); self.seg=seg_len; self.n_seg=L//seg_len; self.kernel=kernel
        self.D=1+n_time+(n_weather if use_weather else 0)+1
        self.seg_embed=nn.Linear(seg_len,d_model)
        self.pos=nn.Parameter(torch.randn(1,self.D,self.n_seg,d_model)*0.02)
        self.tsa=nn.ModuleList([TwoStageAttentionLayer(d_model,heads,self.n_seg,self.D,n_routers,dropout=dropout) for _ in range(layers)])
        self.pool=nn.Linear(self.D*self.n_seg*d_model, d_model)
        self.fut_proj=nn.Sequential(nn.Linear(H*(n_time+(n_weather if use_weather else 0)+1),d_model),nn.GELU())
        self.fuse_norm=nn.LayerNorm(2*d_model)
        self.hour_heads=nn.ModuleList([nn.Linear(2*d_model,2) for _ in range(H)])
        self.trend_lin=nn.Linear(L,H)
        self.seasonal_lin=nn.Linear(L,H)
        self.temp_idx=temp_idx
        if use_weather:
            if temp_idx is None: self.temp_proxy=nn.Linear(n_weather,1)
            self.Tbal_heat=nn.Parameter(torch.tensor(-0.3)); self.Tbal_cool=nn.Parameter(torch.tensor(0.3))
            self.dd_mlp=nn.Sequential(nn.Linear(2,dh),nn.GELU(),nn.Linear(dh,1))
    def _degree_day(self, exo):
        if not self.use_weather: return None
        weather=exo[...,self.n_time:self.n_time+self.n_weather]
        temp = weather[...,self.temp_idx] if self.temp_idx is not None else self.temp_proxy(weather).squeeze(-1)
        temp_fut=temp[:,self.L:]
        heat=F.relu(self.Tbal_heat-temp_fut); cool=F.relu(temp_fut-self.Tbal_cool)
        return self.dd_mlp(torch.stack([heat,cool],-1)).squeeze(-1)
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        cal=exo[:,:self.L,:self.n_time].transpose(1,2)
        parts=[yn.unsqueeze(1), cal]
        if self.use_weather:
            w=exo[:,:self.L,self.n_time:self.n_time+self.n_weather].transpose(1,2); parts.append(w)
        bt=exo[:,:self.L,-1:].transpose(1,2); parts.append(bt)
        allvar=torch.cat(parts,1)
        segs=allvar.reshape(B,self.D,self.n_seg,self.seg)
        z=self.seg_embed(segs)+self.pos
        for layer in self.tsa: z=layer(z)
        pooled=self.pool(z.reshape(B,-1))
        fut=self.fut_proj(exo[:,self.L:].reshape(B,-1))
        fused=self.fuse_norm(torch.cat([pooled,fut],-1))
        out=torch.stack([head(fused) for head in self.hour_heads],1)
        mu_n, raw = out[...,0], out[...,1]
        tr=F.avg_pool1d(F.pad(yn.unsqueeze(1),(self.kernel//2,self.kernel//2),mode='replicate'),self.kernel,1).squeeze(1)
        se=yn-tr
        skip = self.trend_lin(tr) + self.seasonal_lin(se)
        mu_n = mu_n + skip
        corr=self._degree_day(exo)
        if corr is not None: mu_n=mu_n+corr
        return mu_n, raw, mu, sd

class CrossformerLoad_VarSeg(nn.Module):
    def __init__(self, L=168,H=24,n_time=6,n_weather=7,use_weather=True,
                 d_model=256,heads=8,layers=3,seg_load=12,seg_other=24,n_routers=4,dropout=0.15,
                 temp_idx=None,dh=128,**kw):
        super().__init__()
        self.L,self.H=L,H; self.n_time,self.n_weather,self.use_weather=n_time,n_weather,use_weather
        self.revin=RevIN(); self.seg_load,self.seg_other=seg_load,seg_other
        self.n_seg_l, self.n_seg_o = L//seg_load, L//seg_other
        self.Do=n_time+(n_weather if use_weather else 0)+1
        self.load_embed=nn.Linear(seg_load,d_model); self.load_pos=nn.Parameter(torch.randn(1,1,self.n_seg_l,d_model)*0.02)
        self.load_tsa=nn.ModuleList([TwoStageAttentionLayer(d_model,heads,self.n_seg_l,1,n_routers=1,dropout=dropout) for _ in range(layers)])
        self.load_pool=nn.Linear(self.n_seg_l*d_model,d_model)
        self.other_embed=nn.Linear(seg_other,d_model); self.other_pos=nn.Parameter(torch.randn(1,self.Do,self.n_seg_o,d_model)*0.02)
        self.other_tsa=nn.ModuleList([TwoStageAttentionLayer(d_model,heads,self.n_seg_o,self.Do,n_routers,dropout=dropout) for _ in range(layers)])
        self.other_pool=nn.Linear(self.Do*self.n_seg_o*d_model,d_model)
        self.fut_proj=nn.Sequential(nn.Linear(H*(n_time+(n_weather if use_weather else 0)+1),d_model),nn.GELU())
        self.fuse_norm=nn.LayerNorm(3*d_model)
        self.hour_heads=nn.ModuleList([nn.Linear(3*d_model,2) for _ in range(H)])
        self.temp_idx=temp_idx
        if use_weather:
            if temp_idx is None: self.temp_proxy=nn.Linear(n_weather,1)
            self.Tbal_heat=nn.Parameter(torch.tensor(-0.3)); self.Tbal_cool=nn.Parameter(torch.tensor(0.3))
            self.dd_mlp=nn.Sequential(nn.Linear(2,dh),nn.GELU(),nn.Linear(dh,1))
    def _degree_day(self, exo):
        if not self.use_weather: return None
        weather=exo[...,self.n_time:self.n_time+self.n_weather]
        temp = weather[...,self.temp_idx] if self.temp_idx is not None else self.temp_proxy(weather).squeeze(-1)
        temp_fut=temp[:,self.L:]
        heat=F.relu(self.Tbal_heat-temp_fut); cool=F.relu(temp_fut-self.Tbal_cool)
        return self.dd_mlp(torch.stack([heat,cool],-1)).squeeze(-1)
    def forward(self, yh, exo):
        B=yh.size(0); yn,mu,sd=self.revin(yh)
        ls = yn.reshape(B,1,self.n_seg_l,self.seg_load)
        zl = self.load_embed(ls)+self.load_pos
        for layer in self.load_tsa: zl=layer(zl)
        load_pooled = self.load_pool(zl.reshape(B,-1))
        cal=exo[:,:self.L,:self.n_time].transpose(1,2); parts=[cal]
        if self.use_weather:
            w=exo[:,:self.L,self.n_time:self.n_time+self.n_weather].transpose(1,2); parts.append(w)
        bt=exo[:,:self.L,-1:].transpose(1,2); parts.append(bt)
        other_var=torch.cat(parts,1)
        os_ = other_var.reshape(B,self.Do,self.n_seg_o,self.seg_other)
        zo = self.other_embed(os_)+self.other_pos
        for layer in self.other_tsa: zo=layer(zo)
        other_pooled = self.other_pool(zo.reshape(B,-1))
        fut=self.fut_proj(exo[:,self.L:].reshape(B,-1))
        fused=self.fuse_norm(torch.cat([load_pooled,other_pooled,fut],-1))
        out=torch.stack([head(fused) for head in self.hour_heads],1)
        mu_n, raw = out[...,0], out[...,1]
        corr=self._degree_day(exo)
        if corr is not None: mu_n=mu_n+corr
        return mu_n, raw, mu, sd

ITER3_KW = {
  "crossformerload_multiscale":  dict(d_model=224, heads=8, layers=3, seg_lens=(12,24), n_routers=4, dropout=0.15, temp_idx=0, dh=128),
  "crossformerload_dlinearskip": dict(d_model=288, heads=8, layers=4, seg_len=24, n_routers=4, dropout=0.15, temp_idx=0, dh=128),
  "crossformerload_varseg":      dict(d_model=256, heads=8, layers=3, seg_load=12, seg_other=24, n_routers=4, dropout=0.15, temp_idx=0, dh=128),
}
REG["crossformerload_multiscale"] = CrossformerLoad_Multiscale
REG["crossformerload_dlinearskip"] = CrossformerLoad_DLinearSkip
REG["crossformerload_varseg"] = CrossformerLoad_VarSeg
MODEL_KW.update(ITER3_KW)
ALL_MODELS = ALL_MODELS + list(ITER3_KW.keys())
print(f"models now: {ALL_MODELS}")

import csv, time, pandas as pd, pickle, os, traceback
FIELDS=["model","condition","weather","com_nrmse","res_nrmse","com_nmae","res_nmae","com_crps","res_crps","val_nll","params","sec"]
done=set()
if os.path.getsize(SIM_CSV)>50:
    d=pd.read_csv(SIM_CSV); done={(r.model,r.condition) for _,r in d.iterrows()}
PERBUILDING_DIR=f"{RESULTS_DIR}/perbuilding_sim"; os.makedirs(PERBUILDING_DIR,exist_ok=True)

for name in list(ITER3_KW.keys()):
    for cond in CONDITIONS:
        if (name,cond) in done: print(f"skip {name}/{cond}"); continue
        use_w=CONDITIONS[cond]; t0=time.time()
        try:
            torch.manual_seed(SEED)
            base=build(name, L=L,H=H,n_time=TR["Ft"],n_weather=TR["Fw"],use_weather=use_w,**MODEL_KW[name]).to(DEV)
            params=count_params(base)
            print(f"train {name}/{cond} ({params:,}p)...", flush=True)
            val=train(base, base, TR, DEV, use_w, epochs=EPOCHS, patience=PATIENCE, steps=STEPS, bs=BS, lr=LR,
                      amp=USE_AMP, amp_dtype=AMP_DTYPE, clip=1.0, log=True)
            torch.save(base.state_dict(), f"{WEIGHTS_DIR}/{name}_{cond}.pt")
            res=evaluate(base, TE_SIM, DEV, use_w, amp=USE_AMP, amp_dtype=AMP_DTYPE)
            pickle.dump({k:res[k] for k in ("_nrmse","_nmae","_crps","_is_res")},
                        open(f"{PERBUILDING_DIR}/{name}_{cond}.pkl","wb"))
            row={"model":name,"condition":cond,"weather":use_w,
                 "com_nrmse":round(res.get("Com NRMSE",float('nan')),3),"res_nrmse":round(res.get("Res NRMSE",float('nan')),3),
                 "com_nmae":round(res.get("Com NMAE",float('nan')),3),"res_nmae":round(res.get("Res NMAE",float('nan')),3),
                 "com_crps":round(res.get("Com CRPS",float('nan')),3),"res_crps":round(res.get("Res CRPS",float('nan')),3),
                 "val_nll":(round(val,4) if val==val else ""),"params":params,"sec":round(time.time()-t0)}
            csv.DictWriter(open(SIM_CSV,"a",newline=""),FIELDS).writerow(row)
            print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}\n")
        except Exception as e:
            print(f"  FAILED {name}/{cond}: {type(e).__name__}: {e}"); traceback.print_exc()
print("simulated-eval DONE ->", SIM_CSV)

sim = pd.read_csv(SIM_CSV)
print("\n=== all crossformerload variants (condition A) ===")
for m in ["crossformerload","crossformerload_final","crossformerload_multiscale","crossformerload_dlinearskip","crossformerload_varseg"]:
    row = sim[(sim.model==m)&(sim.condition=="A")]
    if len(row): print(f"  {m:28s} Com {row.iloc[0].com_nrmse:7.3f} | Res {row.iloc[0].res_nrmse:7.3f}")


models now: ['pwfnet', 'xpan', 'freqmoe', 'drdt', 'mscah', 'naive', 'linreg', 'lstm', 'gru', 'cnn_lstm', 'dlinear', 'persistence_ensemble', 'xgboost', 'megaloadnet', 'timexerload', 'crossformerload', 'timesnetxl_full', 'timesnetdeep', 'timexerperiodic', 'crossperiodic', 'crossformerload_xl', 'vanillatransformerload', 'vanillatransformerload']
train vanillatransformerload/A (4,358,448p)...
    ep000 val=0.8661 best=0.8661 bad=0
    ep010 val=0.6793 best=0.6793 bad=0
    ep020 val=0.6499 best=0.6499 bad=0
    ep030 val=0.6559 best=0.5977 bad=3
    ep040 val=0.6874 best=0.5765 bad=6
    ep050 val=0.5749 best=0.5438 bad=7
    ep060 val=0.6093 best=0.5438 bad=17
    ep070 val=0.4881 best=0.4769 bad=5
    ep080 val=0.5196 best=0.4769 bad=15
    early stop @ep85
  done 1118s  Com NRMSE=19.323  Res NRMSE=44.868

train vanillatransformerload/B (4,401,456p)...
    ep000 val=0.8874 best=0.8874 bad=0
    ep010 val=0.6837 best=0.6767 bad=2
    ep020 val=0.6582 best=0.6317 bad=2
    ep030 val=0.5824